# Entrenamiento RRNN Tabular 

## Preparación final de dataset para entrenamiento

Utilizaremos `One-hot Encoding` para evitar que la IA se invente jerarquías matemáticas con los barrios o los tipos de habitación, lo que hacemos es "explotar" esa columna de texto en varias columnas binarias nuevas

De esta forma, la IA entiende perfectamente que son categorías independientes. O lo es (1), o no lo es (0). No hay jerarquías falsas.

Debemos comprobar las columnas categóricas que contengan texto, antes del one-hot encoding

In [1]:
import pandas as pd

df = pd.read_csv('../data/listingV5.csv')

columnas_peligrosas = ['id', 'price_per_person', 'estimated_revenue_l365d', 'estimated_occupancy_l365d']
df_limpio = df.drop(columns=columnas_peligrosas, errors='ignore')

X = df_limpio.drop(columns=['price'])

print("=" * 60)
print("COLUMNAS POR TIPO DE DATO:")
print("=" * 60)
print("\n🔤 OBJECT (texto/categorías):")
obj_cols = X.select_dtypes(include='object').columns.tolist()
for col in obj_cols:
    n_unique = X[col].nunique()
    ejemplo = X[col].dropna().iloc[0] if not X[col].dropna().empty else "N/A"
    print(f"  - {col:<45} | únicos: {n_unique:<5} | ej: {str(ejemplo)[:40]}")

print("\n🔢 NUMÉRICO (int/float):")
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
for col in num_cols:
    print(f"  - {col}")

print("\n✅ BOOL:")
bool_cols = X.select_dtypes(include='bool').columns.tolist()
for col in bool_cols:
    print(f"  - {col}")

print(f"\nTotal columnas: {X.shape[1]}")

COLUMNAS POR TIPO DE DATO:

🔤 OBJECT (texto/categorías):
  - host_response_time                            | únicos: 4     | ej: within an hour
  - neighbourhood_cleansed                        | únicos: 11    | ej: Este
  - property_type                                 | únicos: 39    | ej: Entire rental unit
  - room_type                                     | únicos: 4     | ej: Entire home/apt

🔢 NUMÉRICO (int/float):
  - host_response_rate
  - host_is_superhost
  - host_listings_count
  - latitude
  - longitude
  - accommodates
  - bathrooms
  - bedrooms
  - beds
  - minimum_nights
  - maximum_nights
  - availability_365
  - number_of_reviews
  - number_of_reviews_ltm
  - review_scores_rating
  - review_scores_accuracy
  - review_scores_cleanliness
  - review_scores_checkin
  - review_scores_communication
  - review_scores_location
  - review_scores_value
  - instant_bookable
  - reviews_per_month
  - private_bathroom
  - has_kitchen
  - has_hair_dryer
  - has_iron
  - has_bed_line

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
import joblib

# =============================================================================
# PASO 1: CARGA DEL DATASET
# =============================================================================
# Cargamos el CSV final que salió del EDA (ya limpio, sin nulos, sin outliers)
df = pd.read_csv('../data/listingV5.csv')


# =============================================================================
# PASO 2: PURGA DE DATA LEAKAGE (Fuga de Datos)
# =============================================================================
# Data Leakage = incluir en el entrenamiento información que NO existiría
# en el momento real de hacer la predicción. Es el error más grave en ML.
#
# - 'id'                      → Número de serie, no tiene relación causal con el precio
# - 'price_per_person'        → LEAKAGE DIRECTO: se calcula dividiendo 'price' entre
#                               los huéspedes. La red aprendería "trampa" en lugar de
#                               aprender patrones reales del mercado.
# - 'estimated_revenue_l365d' → LEAKAGE DIRECTO: es price * días ocupados. Contiene
#                               el precio de forma implícita.
# - 'estimated_occupancy_l365d'→ Se deriva también de los ingresos estimados.
#
# errors='ignore' evita que el código falle si alguna de estas columnas ya no
# existe en el CSV (por ejemplo si el dataset se actualiza en el futuro).
columnas_peligrosas = ['id', 'price_per_person', 'estimated_revenue_l365d', 'estimated_occupancy_l365d']
df_limpio = df.drop(columns=columnas_peligrosas, errors='ignore')


# =============================================================================
# PASO 3: SEPARAR FEATURES (X) DEL TARGET (y)
# =============================================================================
# X = todo lo que la red neuronal VERÁ para hacer su predicción (las "pistas")
# y = lo que la red neuronal debe APRENDER A PREDECIR (el precio por noche)
#
# Esta separación es fundamental: el modelo nunca debe "ver" el precio
# durante el entrenamiento excepto para calcular el error y ajustar los pesos.
X = df_limpio.drop(columns=['price'])
y = df_limpio['price']


# =============================================================================
# PASO 4: AGRUPAR CATEGORÍAS MINORITARIAS EN 'property_type'
# =============================================================================
# El diagnóstico reveló que 'property_type' tiene 39 categorías únicas.
# Si hacemos One-Hot Encoding directo, generaríamos 38 columnas nuevas,
# muchas de ellas con muy pocos ejemplos (ej: "Yurt" o "Houseboat" con 2 casos).
#
# Problema: una columna con 2 unos en 5000 filas no aporta información útil
# a la red; solo añade ruido y hace el entrenamiento más inestable.
#
# Solución: nos quedamos con los 10 tipos más frecuentes (que representan
# la gran mayoría del dataset) y todo lo demás pasa a llamarse "Other".
# Así controlamos la dimensionalidad sin perder información relevante.
TOP_N_PROPERTY = 10
top_property_types = X['property_type'].value_counts().nlargest(TOP_N_PROPERTY).index
X['property_type'] = X['property_type'].where(
    X['property_type'].isin(top_property_types),  # Si está en el top 10, se mantiene
    other='Other'                                   # Si no, se reemplaza por 'Other'
)

print("Distribución property_type tras agrupación:")
print(X['property_type'].value_counts())


# =============================================================================
# PASO 5: CLASIFICAR COLUMNAS POR TIPO DE TRATAMIENTO
# =============================================================================
# Definimos explícitamente qué columnas son categóricas (texto con categorías)
# para que el ColumnTransformer sepa qué transformación aplicar a cada una.
# El resto de columnas (numéricas) se calcula automáticamente por descarte.
categorical_cols = ['host_response_time', 'neighbourhood_cleansed', 'room_type', 'property_type']
numerical_cols   = [col for col in X.columns if col not in categorical_cols]

print(f"\nColumnas numéricas ({len(numerical_cols)}): {numerical_cols}")


# =============================================================================
# PASO 6: COLUMNTRANSFORMER — EL "DIRECTOR DE ORQUESTA" DEL PREPROCESADO
# =============================================================================
# ColumnTransformer aplica transformaciones DIFERENTES a grupos de columnas
# distintos, todo dentro de un único objeto. Es la forma correcta en Scikit-Learn
# de manejar datasets mixtos (números + texto).
#
# Transformación 1 — StandardScaler para numéricas:
#   Convierte cada columna numérica para que tenga media=0 y desviación=1.
#   Ejemplo: si 'price' va de 20€ a 500€ y 'bathrooms' va de 1 a 5,
#   sin escalar el gradiente "perseguiría" más al precio que a los baños.
#   Con scaling, todas las variables tienen el mismo "peso matemático" inicial.
#
# Transformación 2 — OneHotEncoder para categóricas:
#   Convierte texto en columnas binarias (0 o 1).
#   Ejemplo: room_type='Entire home/apt' → [1, 0, 0, 0]
#            room_type='Private room'    → [0, 1, 0, 0]
#   La red neuronal no puede operar con texto, solo con números.
#
#   · handle_unknown='ignore': Si en producción llega un barrio nuevo que
#     el modelo nunca vio durante el entrenamiento, en lugar de lanzar un
#     error, simplemente pone todos los bits a 0 (categoría desconocida).
#     Crítico para que la API no se caiga en producción.
#
#   · sparse_output=False: Por defecto OneHotEncoder devuelve una matriz
#     "sparse" (dispersa, formato comprimido). PyTorch necesita arrays
#     densos de NumPy, así que forzamos el formato completo.
#
# remainder='drop':
#   Si hubiera alguna columna que no está en numerical_cols ni en
#   categorical_cols (por un despiste), la elimina en lugar de lanzar error.
#   Es la red de seguridad del pipeline.

## ATENCIÓN!! En la primera ejecución de la RRNN nos da error ya que aún existen algunos valores Nan en el dataset
# Por ello deberemos sustituir los valores NaN por la mediana de cada columna numérica, y por la categoría más frecuente en las columnas categóricas.
# Columnas que necesitan imputación por mediana (numéricas continuas)
# El resto de numéricas no tienen NaN pero las incluimos igualmente
# para que el pipeline sea robusto ante datos nuevos en producción

# --- Pipeline para variables numéricas ---
# Pipeline encadena transformaciones en orden: primero imputa, luego escala.
# Es importante este orden: si escalamos antes de imputar, los NaN
# se propagan al scaler y obtenemos el mismo problema.
numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),  # Rellena NaN con la mediana del Train
    ('scaler',  StandardScaler())                   # Escala a media=0, desviación=1
])

# --- Pipeline para variables categóricas ---
# Añadimos también imputación categórica por si hubiera NaN en texto
# strategy='most_frequent' rellena con la categoría más común
categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# --- ColumnTransformer actualizado ---
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_pipeline,      numerical_cols),
        ('cat', categorical_pipeline,  categorical_cols)
    ],
    remainder='drop'
)


# =============================================================================
# PASO 7: DIVISIÓN EN TRES CONJUNTOS (Train / Validation / Test)
# =============================================================================
# Por qué necesitamos TRES conjuntos y no dos:
#
# · TRAIN (64%): El modelo VE estos datos y ajusta sus pesos con ellos.
#
# · VALIDATION (16%): El modelo NUNCA aprende de estos datos.
#   Se usan DURANTE el entrenamiento para medir si el modelo está
#   generalizando o memorizando (overfitting). También es el conjunto
#   que usará el Early Stopping para decidir cuándo parar.
#
# · TEST (20%): El modelo NUNCA los ha visto en ningún momento.
#   Solo se usan AL FINAL para obtener las métricas definitivas (R², MAE).
#   Simula un apartamento completamente nuevo que llega a la API.
#
# La división se hace en dos pasos:
#   1º → Separamos el Test (20%) del resto (80%)
#   2º → Del 80% restante, separamos Validation (20% de ese 80% = 16% del total)
#        lo que nos deja Train con el 80% de ese 80% = 64% del total.
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42   # random_state=42 → reproducibilidad
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.20, random_state=42
)


# =============================================================================
# PASO 8: FIT + TRANSFORM (La regla de oro del preprocesado)
# =============================================================================
# REGLA CRÍTICA: el preprocessor solo puede hacer .fit() sobre datos de Train.
#
# .fit_transform(X_train):
#   · fit()      → Aprende los parámetros: media y desviación de cada columna
#                  numérica, y qué categorías existen en las columnas de texto.
#   · transform()→ Aplica esas transformaciones al propio Train.
#
# .transform(X_val) y .transform(X_test):
#   · Solo aplica los parámetros YA APRENDIDOS del Train. No recalcula nada.
#   · Esto es correcto porque en producción tampoco recalcularíamos: usaríamos
#     los parámetros del entrenamiento para normalizar cada nuevo apartamento.
#
# Si hiciéramos fit() también en Validation/Test, estaríamos "filtrando" 
# información futura al modelo (otro tipo de Data Leakage más sutil).
X_train_scaled = preprocessor.fit_transform(X_train)
X_val_scaled   = preprocessor.transform(X_val)
X_test_scaled  = preprocessor.transform(X_test)


# =============================================================================
# PASO 9: CONVERTIR TARGETS A FLOAT32
# =============================================================================
# PyTorch trabaja internamente con tensores de tipo float32 (32 bits).
# Pandas usa float64 (64 bits) por defecto.
# Convertimos ahora para evitar errores de tipo en el DataLoader de PyTorch.
y_train = y_train.to_numpy(dtype=np.float32)
y_val   = y_val.to_numpy(dtype=np.float32)
y_test  = y_test.to_numpy(dtype=np.float32)


# =============================================================================
# PASO 10: GUARDAR EL PREPROCESSOR PARA PRODUCCIÓN
# =============================================================================

#Por qué no guardar en CSV entonces
#Un CSV solo guarda los datos ya transformados, pero no guarda la receta de cómo transformarlos.

# Con joblib.dump() serializamos el objeto preprocessor a disco.
# Cuando despleguemos la API (FastAPI/Flask), haremos joblib.load() para
# recuperarlo y aplicar exactamente la misma transformación a los datos
# que lleguen de los usuarios, garantizando consistencia total.
import os
os.makedirs('../data', exist_ok=True)
joblib.dump(preprocessor, '../data/preprocessor.pkl')


# =============================================================================
# RESUMEN FINAL
# =============================================================================
print(f"\n✅ Train:      {X_train_scaled.shape}")
print(f"✅ Validation: {X_val_scaled.shape}")
print(f"✅ Test:       {X_test_scaled.shape}")
print(f"\n🧠 Variables de entrada al modelo: {X_train_scaled.shape[1]}")
print("¡Datos listos para PyTorch!")

Distribución property_type tras agrupación:
property_type
Entire rental unit             4129
Private room in rental unit     403
Entire condo                    228
Entire home                     225
Entire serviced apartment       218
Other                           217
Entire loft                     168
Private room in home             87
Entire villa                     42
Entire vacation home             42
Entire townhouse                 37
Name: count, dtype: int64

Columnas numéricas (44): ['host_response_rate', 'host_is_superhost', 'host_listings_count', 'latitude', 'longitude', 'accommodates', 'bathrooms', 'bedrooms', 'beds', 'minimum_nights', 'maximum_nights', 'availability_365', 'number_of_reviews', 'number_of_reviews_ltm', 'review_scores_rating', 'review_scores_accuracy', 'review_scores_cleanliness', 'review_scores_checkin', 'review_scores_communication', 'review_scores_location', 'review_scores_value', 'instant_bookable', 'reviews_per_month', 'private_bathroom', 'has_k

## Activar GPU NVIDIA 

In [3]:
# pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121
import torch

print(torch.__version__)               # Debe contener "cu121" o similar, NO "cpu"
print(torch.cuda.is_available())       # Debe imprimir: True
print(torch.cuda.get_device_name(0))   # Debe imprimir el nombre de tu GPU, ej: "NVIDIA GeForce RTX 3060"
device = (
    "cuda"  if torch.cuda.is_available() else
    "mps"   if torch.backends.mps.is_available() else  # Apple Silicon
    "cpu"
)
print(f"Usando dispositivo: {device}")
# Output esperado en tu caso → "Usando dispositivo: cuda"

2.5.1
True
NVIDIA GeForce RTX 4070 Laptop GPU
Usando dispositivo: cuda


## Clase AirbnbDataset

PyTorch no puede entrenar directamente sobre arrays de NumPy. Necesita dos cosas:

Dataset: un objeto que sepa "tengo N ejemplos, y el ejemplo número i es este par (X, y)"

DataLoader: un objeto que use el Dataset para entregar los datos en lotes (batches) durante el entrenamiento, mezclados aleatoriamente en cada época

In [4]:
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np

# =============================================================================
# CLASE AirbnbDataset — Envuelve nuestros arrays NumPy en formato PyTorch
# =============================================================================
# Para crear un Dataset personalizado en PyTorch SIEMPRE hay que:
# 1. Heredar de la clase base 'Dataset'
# 2. Implementar obligatoriamente estos tres métodos:
#    - __init__  → constructor, guarda los datos
#    - __len__   → cuántos ejemplos tiene el dataset
#    - __getitem__→ cómo obtener el ejemplo número 'idx'
#
# El DataLoader llamará a __getitem__ e __len__ automáticamente. Nunca debemos llamarlo directamente

class AirbnbDataset(Dataset):

    def __init__(self, X, y):
        # Convertimos los arrays NumPy a tensores de PyTorch.
        # float32 es el tipo estándar para redes neuronales (float64 consume
        # el doble de memoria en GPU sin aportar precisión relevante).
        #
        # X ya es un array NumPy (salida del ColumnTransformer)
        # y ya es un array NumPy float32 (lo convertimos antes con to_numpy)
        self.X = torch.tensor(X, dtype=torch.float32)
        
        # .unsqueeze(1) convierte y de forma [N] a forma [N, 1]
        # Ejemplo: [120.0, 85.0, 200.0] → [[120.0], [85.0], [200.0]]
        # Esto es necesario porque la última capa de la red también
        # devuelve forma [N, 1], y PyTorch exige que ambos tengan
        # la misma forma para calcular el error (MSELoss).
        self.y = torch.tensor(y, dtype=torch.float32).unsqueeze(1)

    def __len__(self):
        # Devuelve el número total de ejemplos del dataset.
        # El DataLoader lo usa para saber cuántos batches hay por época.
        return len(self.X)

    def __getitem__(self, idx):
        # Dado un índice, devuelve el par (features, precio) correspondiente.
        # El DataLoader llama a este método batch_size veces por cada lote.
        return self.X[idx], self.y[idx]


# =============================================================================
# CREAR LOS TRES DATASETS
# =============================================================================
# Simplemente instanciamos la clase con cada split que preparamos antes.

dataset_train = AirbnbDataset(X_train_scaled, y_train)
dataset_val   = AirbnbDataset(X_val_scaled,   y_val)
dataset_test  = AirbnbDataset(X_test_scaled,  y_test)


# =============================================================================
# CREAR LOS TRES DATALOADERS
# =============================================================================
# El DataLoader es el que realmente alimenta la red durante el entrenamiento.
#
# Parámetros clave:
#
# · batch_size=64:
#   En lugar de pasar los ~4000 apartamentos de golpe (lo que saturaría
#   la memoria y daría un gradiente muy "promediado"), los pasamos de 64
#   en 64. Cada lote actualiza los pesos una vez.
#   64 es un buen balance entre velocidad y estabilidad del gradiente.
#
# · shuffle=True (solo en Train):
#   En cada época, baraja el orden de los apartamentos antes de formar
#   los batches. Esto evita que la red memorice el orden de los datos
#   y mejora la generalización.
#   En Validation y Test NO se baraja porque solo estamos evaluando,
#   el orden no afecta al resultado.
#
# · num_workers=0:
#   Número de procesos paralelos para cargar datos. En Windows, usar
#   num_workers > 0 dentro de un notebook causa errores de multiproceso,
#   así que lo dejamos en 0 (carga en el proceso principal).
#   En Linux/Mac se puede subir a 2 o 4 para más velocidad.

BATCH_SIZE = 64

loader_train = DataLoader(dataset_train, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
loader_val   = DataLoader(dataset_val,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
loader_test  = DataLoader(dataset_test,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)


# =============================================================================
# VERIFICACIÓN — Comprobamos que todo tiene la forma correcta
# =============================================================================
# Extraemos un batch de prueba del loader de train para inspeccionarlo.
X_batch_ejemplo, y_batch_ejemplo = next(iter(loader_train))

print(f"Tamaño del dataset Train:      {len(dataset_train)} apartamentos")
print(f"Tamaño del dataset Validation: {len(dataset_val)} apartamentos")
print(f"Tamaño del dataset Test:       {len(dataset_test)} apartamentos")
print(f"\nForma de un batch de X: {X_batch_ejemplo.shape}")  # [64, num_features]
print(f"Forma de un batch de y: {y_batch_ejemplo.shape}")  # [64, 1]
print(f"\nNúmero de batches por época (train): {len(loader_train)}")
print(f"Tipo de tensor X: {X_batch_ejemplo.dtype}")
print(f"Tipo de tensor y: {y_batch_ejemplo.dtype}")

Tamaño del dataset Train:      3708 apartamentos
Tamaño del dataset Validation: 928 apartamentos
Tamaño del dataset Test:       1160 apartamentos

Forma de un batch de X: torch.Size([64, 73])
Forma de un batch de y: torch.Size([64, 1])

Número de batches por época (train): 58
Tipo de tensor X: torch.float32
Tipo de tensor y: torch.float32


## RRNN Tabular V1

### Construcción RRNN

In [5]:
import torch.nn as nn


# =============================================================================
# ARQUITECTURA DE LA RED NEURONAL — AirbnbMLP
# =============================================================================
# MLP = Multi-Layer Perceptron (Perceptrón Multicapa)
# Es la arquitectura más simple de red neuronal densa: cada neurona de una
# capa está conectada con TODAS las neuronas de la capa siguiente.
#
# Para crear un modelo en PyTorch SIEMPRE hay que:
# 1. Heredar de nn.Module (la clase base de todos los modelos PyTorch)
# 2. Definir las capas en __init__
# 3. Definir el flujo de datos en forward()


class AirbnbMLP(nn.Module):


    def __init__(self, input_size):
        # Llamamos al constructor del padre (obligatorio en PyTorch)
        super(AirbnbMLP, self).__init__()


        # -----------------------------------------------------------------
        # DEFINICIÓN DE CAPAS
        # -----------------------------------------------------------------
        # nn.Sequential agrupa capas en orden. Los datos entran por la
        # primera y salen por la última, en secuencia automática.
        # Es más limpio que definir cada capa por separado.


        self.network = nn.Sequential(
            # CAPA 1: Capa de entrada → 128 neuronas
            nn.Linear(input_size, 128),  # capa densa con 128 neuronas
            nn.BatchNorm1d(128),         # Normaliza las activaciones entre capas
            nn.ReLU(),                   # Función de activación que introduce no-linealidad. Permite a la red aprender patrones complejos.
            nn.Dropout(p=0.2),           # Durante el entrenamiento, "apaga" aleatoriamente el 20% de las neuronas en cada forward pass.
                                         # Esto fuerza a la red a no depender de neuronas específicas y a aprender representaciones más robustas.
                                         # Resultado: menos overfitting (sobreajuste).
            # CAPA 2: 128 → 64 neuronas
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(p=0.2),
            # CAPA 3: 64 → 32 neuronas
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(p=0.2),
            # CAPA DE SALIDA: 32 → 1 neurona
            # Una sola neurona porque es un problema de REGRESIÓN:
            # queremos predecir UN número (el precio por noche).
            # Sin función de activación final → salida lineal ilimitada,
            # lo que permite predecir cualquier valor positivo o negativo.
            # (Si fuera clasificación usaríamos Softmax o Sigmoid aquí)
            nn.Linear(32, 1)
        )


    def forward(self, x):
        # forward() define cómo fluyen los datos a través de la red.
        # PyTorch llama a este método automáticamente cuando hacemos
        # predicciones: model(X_batch) → internamente ejecuta forward(X_batch)
        #
        # En nuestro caso es simple: los datos entran y recorren
        # todas las capas del Sequential en orden.
        return self.network(x)


# =============================================================================
# INSTANCIAR EL MODELO Y MOVERLO A LA GPU
# =============================================================================
# .to(device) mueve TODOS los parámetros del modelo (pesos y biases)
# a la GPU. A partir de aquí, los cálculos ocurren en la gráfica.
INPUT_SIZE = X_train_scaled.shape[1]
model = AirbnbMLP(input_size=INPUT_SIZE).to(device)


# =============================================================================
# INSPECCIÓN DEL MODELO
# =============================================================================
# print(model) muestra la arquitectura completa con todas las capas.
# Útil para verificar que la estructura es la esperada.
print(model)
print(f"\nDispositivo del modelo: {next(model.parameters()).device}")
print(f"INPUT_SIZE detectado automáticamente: {INPUT_SIZE}")

# Contamos el total de parámetros entrenables (pesos + biases)
# Es una métrica estándar para medir la "capacidad" de la red.
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parámetros entrenables: {total_params:,}")

AirbnbMLP(
  (network): Sequential(
    (0): Linear(in_features=73, out_features=128, bias=True)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.2, inplace=False)
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.2, inplace=False)
    (8): Linear(in_features=64, out_features=32, bias=True)
    (9): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): Dropout(p=0.2, inplace=False)
    (12): Linear(in_features=32, out_features=1, bias=True)
  )
)

Dispositivo del modelo: cuda:0
INPUT_SIZE detectado automáticamente: 73
Total parámetros entrenables: 20,289


### Entrenamiento

In [6]:
import torch.nn as nn
import torch.optim as optim
import copy
import numpy as np


# =============================================================================
# CONFIGURACIÓN DE HIPERPARÁMETROS
# =============================================================================
LEARNING_RATE = 0.001  # Tasa de aprendizaje del optimizador Adam
EPOCHS        = 5000   # Número máximo de épocas (el Early Stopping parará antes)
PATIENCE      = 30     # Early Stopping: épocas sin mejora antes de parar


# =============================================================================
# FUNCIÓN DE PÉRDIDA Y OPTIMIZADOR
# =============================================================================
# MSELoss (Mean Squared Error): penaliza los errores grandes de forma cuadrática.
# Si el modelo falla por 10€, el error es 100. Si falla por 50€, el error es 2500.
# Esto obliga a la red a evitar errores graves, ideal para precios de Airbnb.
criterion = nn.MSELoss()

# Adam (Adaptive Moment Estimation): optimizador que ajusta automáticamente
# la tasa de aprendizaje por parámetro. Es el estándar en Deep Learning.
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)


# =============================================================================
# LÓGICA DE EARLY STOPPING
# =============================================================================
# Variables de control para detener el entrenamiento si el modelo
# deja de mejorar en el conjunto de validación.
mejor_val_loss     = float('inf')  # Inicializamos con infinito para que cualquier pérdida real sea mejor en la primera época
epocas_sin_mejora  = 0             # Contador de épocas consecutivas sin mejora
mejor_modelo_pesos = None          # Guardaremos aquí los pesos del mejor modelo
mejor_r2           = None          # R² del mejor modelo, para el resumen final


# =============================================================================
# LISTAS PARA GUARDAR EL HISTORIAL DE ENTRENAMIENTO
# =============================================================================
# Las usaremos después para graficar las curvas de aprendizaje
historial_train_loss = []
historial_train_mae  = []
historial_val_loss   = []
historial_val_mae    = []
historial_val_rmse   = []
historial_r2         = []


# =============================================================================
# BUCLE DE ENTRENAMIENTO PRINCIPAL
# =============================================================================
print(f"Iniciando entrenamiento en: {device}")
print(f"{'Época':<8} {'Train MSE':>10} {'Train MAE':>10} {'Val MSE':>10} {'Val MAE':>10} {'Val RMSE':>10} {'R²':>8} {'Estado':>10}")
print("-" * 90)


for epoca in range(1, EPOCHS + 1):


    # -------------------------------------------------------------------------
    # FASE 1: ENTRENAMIENTO (modo train)
    # -------------------------------------------------------------------------
    # model.train() activa el modo entrenamiento:
    # - Dropout está ACTIVO (apaga neuronas aleatoriamente)
    # - Los gradientes se calculan y acumulan
    model.train()
    train_loss_acumulado = 0.0
    train_mae_acumulado  = 0.0


    for X_batch, y_batch in loader_train:

        # Mover los tensores del batch a la GPU en cada iteración.
        # Los datos viven en RAM (CPU) y los movemos a VRAM (GPU) por lotes.
        # No podemos moverlos todos de golpe porque la VRAM tiene límite.
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        # PASO 1: Borrar gradientes acumulados del batch anterior.
        # PyTorch ACUMULA gradientes por defecto, si no los borramos
        # se sumarían a los del batch anterior y los pesos se actualizarían
        # con información incorrecta.
        optimizer.zero_grad()

        # PASO 2: Forward pass — los datos fluyen a través de la red
        # y obtenemos las predicciones del batch actual.
        predicciones = model(X_batch)

        # PASO 3: Calcular el error (Loss) entre predicciones y valores reales.
        loss = criterion(predicciones, y_batch)

        # PASO 4: Backward pass — PyTorch calcula automáticamente el gradiente
        # de la pérdida respecto a CADA parámetro de la red (regla de la cadena).
        # Esto indica en qué dirección y cuánto hay que mover cada peso.
        loss.backward()

        # PASO 5: El optimizador actualiza los pesos usando los gradientes
        # calculados en el backward pass.
        optimizer.step()

        # Acumulamos la pérdida para calcular el promedio de la época.
        # .item() extrae el valor escalar del tensor (necesario para operar en CPU)
        train_loss_acumulado += loss.item()

        # MAE de entrenamiento: permite comparar visualmente con el MAE de validación
        # y detectar overfitting (si train MAE baja pero val MAE sube, hay sobreajuste)
        train_mae_acumulado += torch.mean(torch.abs(predicciones - y_batch)).item()

    # Pérdida media de todos los batches de entrenamiento en esta época
    train_loss_media = train_loss_acumulado / len(loader_train)
    train_mae_media  = train_mae_acumulado  / len(loader_train)


    # -------------------------------------------------------------------------
    # FASE 2: VALIDACIÓN (modo eval)
    # -------------------------------------------------------------------------
    # model.eval() activa el modo evaluación:
    # - Dropout está DESACTIVADO (todas las neuronas activas → predicciones estables)
    # - BatchNorm usa estadísticas fijas
    model.eval()
    val_loss_acumulado = 0.0
    val_mae_acumulado  = 0.0
    val_rmse_acumulado = 0.0

    # Para R² necesitamos acumular TODAS las predicciones y reales del epoch completo
    # No podemos calcularlo batch a batch porque R² necesita la media global
    todas_preds  = []
    todos_reales = []

    # torch.no_grad() desactiva el cálculo de gradientes durante la validación.
    # No necesitamos gradientes porque no vamos a actualizar pesos.
    # Ahorra memoria en GPU y acelera el cálculo ~2x.
    with torch.no_grad():
        for X_batch, y_batch in loader_val:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            predicciones = model(X_batch)

            # MSE Loss para el criterio de Early Stopping
            val_loss_acumulado += criterion(predicciones, y_batch).item()

            # MAE (Mean Absolute Error): fallo medio en €/noche
            # Mucho más intuitivo que el MSE para comunicar resultados.
            # Si MAE = 28 significa que el modelo falla de media 28€/noche.
            val_mae_acumulado += torch.mean(torch.abs(predicciones - y_batch)).item()

            # RMSE (Root Mean Squared Error): raíz del error cuadrático medio.
            # Similar al MAE pero penaliza más los errores grandes.
            # Útil para detectar si el modelo falla mucho en precios extremos.
            val_rmse_acumulado += torch.sqrt(torch.mean((predicciones - y_batch) ** 2)).item()

            # Acumulamos predicciones y reales para calcular R² al final de la época
            todas_preds.extend(predicciones.cpu().numpy().flatten())
            todos_reales.extend(y_batch.cpu().numpy().flatten())

    val_loss_media = val_loss_acumulado / len(loader_val)
    val_mae_media  = val_mae_acumulado  / len(loader_val)
    val_rmse_media = val_rmse_acumulado / len(loader_val)

    # R² (Coeficiente de Determinación):
    # Mide qué porcentaje de la variación del precio explica el modelo.
    # R² = 1 - (suma errores cuadráticos / varianza total del target)
    # R²=1.0 → predicción perfecta
    # R²=0.0 → el modelo no es mejor que predecir siempre la media
    # R²<0.0 → el modelo es peor que predecir siempre la media
    todas_preds  = np.array(todas_preds)
    todos_reales = np.array(todos_reales)
    ss_res = np.sum((todos_reales - todas_preds) ** 2)
    ss_tot = np.sum((todos_reales - np.mean(todos_reales)) ** 2)
    r2 = 1 - (ss_res / ss_tot)

    # Guardamos en el historial para las gráficas posteriores
    historial_train_loss.append(train_loss_media)
    historial_train_mae.append(train_mae_media)
    historial_val_loss.append(val_loss_media)
    historial_val_mae.append(val_mae_media)
    historial_val_rmse.append(val_rmse_media)
    historial_r2.append(r2)


    # -------------------------------------------------------------------------
    # LÓGICA DE EARLY STOPPING
    # -------------------------------------------------------------------------
    if val_loss_media < mejor_val_loss:
        # El modelo mejoró: actualizamos el mejor loss, reseteamos el contador
        # y guardamos una copia de los pesos actuales en memoria.
        # .state_dict() devuelve un diccionario con todos los pesos y biases.
        # copy.deepcopy garantiza que guardamos una copia independiente,
        # no una referencia que se sobreescribiría en la siguiente época.
        mejor_val_loss     = val_loss_media
        epocas_sin_mejora  = 0
        mejor_modelo_pesos = copy.deepcopy(model.state_dict())
        mejor_r2           = r2
        estado = "✅ Mejor"
    else:
        epocas_sin_mejora += 1
        estado = f"⏳ {epocas_sin_mejora}/{PATIENCE}"

    # Imprimir progreso cada 5 épocas (o en la primera)
    if epoca % 5 == 0 or epoca == 1:
        print(f"{epoca:<8} {train_loss_media:>10.2f} {train_mae_media:>10.2f} {val_loss_media:>10.2f} {val_mae_media:>10.2f} {val_rmse_media:>10.2f} {r2:>8.4f} {estado:>10}")

    # Condición de parada: si llevamos PATIENCE épocas sin mejorar, paramos
    if epocas_sin_mejora >= PATIENCE:
        print(f"\n⛔ Early Stopping activado en época {epoca}.")
        print(f"   El modelo no mejoró en {PATIENCE} épocas consecutivas.")
        break


# =============================================================================
# RESTAURAR LOS MEJORES PESOS Y GUARDAR EL MODELO
# =============================================================================
# Cargamos los pesos de la época donde el modelo tuvo menor val_loss.
# Sin esto, el modelo final tendría los pesos de la última época,
# que podrían estar sobreajustados.
model.load_state_dict(mejor_modelo_pesos)

# Guardamos el modelo entrenado a disco para usarlo en la API.
# state_dict() es preferible a guardar el modelo entero porque es
# más portable y no depende de la estructura de clases del proyecto.
torch.save(mejor_modelo_pesos, '../data/airbnb_mlp.pt')

print(f"\n{'='*60}")
print(f"✅ Modelo guardado en '../data/airbnb_mlp.pt'")
print(f"   Mejor Val MSE:  {mejor_val_loss:.2f}")
print(f"   Mejor Val MAE:  {min(historial_val_mae):.2f} €/noche")
print(f"   Mejor Val RMSE: {min(historial_val_rmse):.2f} €/noche")
print(f"   Mejor R²:       {mejor_r2:.4f}  →  El modelo explica el {mejor_r2*100:.1f}% de la variación de precios")

Iniciando entrenamiento en: cuda
Época     Train MSE  Train MAE    Val MSE    Val MAE   Val RMSE       R²     Estado
------------------------------------------------------------------------------------------
1          23908.85     119.30   28795.86     118.27     155.44  -0.8907    ✅ Mejor
5          21793.07     113.83   26509.03     112.34     147.62  -0.7412    ✅ Mejor
10         16306.07      96.55   21038.55      94.20     126.43  -0.3850    ✅ Mejor
15         10608.73      73.41   16989.18      74.37     107.28  -0.1213    ✅ Mejor
20          6896.42      52.36   13595.20      54.29      88.88   0.1010    ✅ Mejor
25          5170.28      41.14   12089.60      40.95      78.28   0.1969    ✅ Mejor
30          4204.72      35.35   11537.87      35.39      73.50   0.2316    ✅ Mejor
35          4171.07      34.78   11149.38      32.91      70.02   0.2573    ✅ Mejor
40          3468.90      32.71   11344.88      32.87      70.89   0.2445     ⏳ 5/30
45          3083.75      31.57   114

### Análisis primera RRNN 

La RRNN actual nos da un MAE de alrededor de 30-32/noche, lo que es un error medio algo alto para el objetivo que tenemos que sería sobre los 15€/noche. Esto puede deberse a que los outliers donde se encuentran apartamentos de 1000€ la noche. La red se "obsesiona" con intentar predecir ese caso extremo y descuida aprender el patrón del 90% de pisos normales.

Se plantean dos alternativas, remover outliers del dataset o transformación logarítmica. Optamos por la transformación logarítmica para mantener estos pisos de lujo en el dataset y no perder esa información por si alguno de los clientes que quieran usar la IA poseen uno de ellos.

Con la transformación logarítmica la red neuronal dará una importancia similar a una distancia entre 50 y 100€ que a otra entre 900 y 1825€. Escalando logarítmicamente es como si dijeras: "un error de 10€ en un piso de 60€ es igual de importante que un error de 200€ en un piso de 1200€".

Esta es una práctica estándar en la industria inmobiliaria ya que no puedes permitirte ignorar el segmento de lujo si tu producto va dirigido a inversores.

## RRNN V2 transformación logarítmica

### Nuevo preprocesamiento con transformación logaritmica

In [7]:
import numpy as np
from torch.utils.data import Dataset, DataLoader
import torch

# =============================================================================
# TRANSFORMACIÓN LOGARÍTMICA DEL TARGET
# =============================================================================
# El precio tiene una distribución muy asimétrica: muchos pisos entre 50-150€
# y pocos pisos con precios extremos (500€, 1825€).
# Esos outliers hacen que el MSE "persiga" los casos raros en lugar de
# aprender el patrón general, disparando la pérdida desde el primer batch.
#
# np.log1p(x) = log(x + 1) transforma la distribución a una campana más
# simétrica y equilibrada. La red aprende en escala logarítmica y luego
# deshacemos la transformación para interpretar los resultados en euros.
#
# IMPORTANTE: log1p se aplica DESPUÉS de to_numpy, ANTES de crear los Datasets.
# Y se guarda y_test SIN transformar para la evaluación final en euros reales.

y_train_log = np.log1p(y_train).astype(np.float32)
y_val_log   = np.log1p(y_val).astype(np.float32)
# y_test lo guardamos en euros reales para la evaluación final
y_test_euros = y_test.copy()  # copia sin transformar para métricas finales

print(f"Rango precios originales:     [{y_train.min():.0f}€, {y_train.max():.0f}€]")
print(f"Rango precios log-escalados:  [{y_train_log.min():.2f}, {y_train_log.max():.2f}]")
print(f"Media log-escalada:           {y_train_log.mean():.2f}")


# =============================================================================
# CLASE AirbnbDataset
# =============================================================================
class AirbnbDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).unsqueeze(1)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


# =============================================================================
# DATASETS Y DATALOADERS — ahora con target log-transformado
# =============================================================================
BATCH_SIZE = 64

dataset_train = AirbnbDataset(X_train_scaled, y_train_log)
dataset_val   = AirbnbDataset(X_val_scaled,   y_val_log)
dataset_test  = AirbnbDataset(X_test_scaled,  y_test_euros)  # test en euros reales

loader_train = DataLoader(dataset_train, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
loader_val   = DataLoader(dataset_val,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
loader_test  = DataLoader(dataset_test,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

X_batch_ejemplo, y_batch_ejemplo = next(iter(loader_train))
print(f"\nForma de un batch de X: {X_batch_ejemplo.shape}")
print(f"Forma de un batch de y: {y_batch_ejemplo.shape}")
print(f"Rango y en batch (log): [{y_batch_ejemplo.min().item():.2f}, {y_batch_ejemplo.max().item():.2f}]")

Rango precios originales:     [19€, 1825€]
Rango precios log-escalados:  [3.00, 7.51]
Media log-escalada:           4.63

Forma de un batch de X: torch.Size([64, 73])
Forma de un batch de y: torch.Size([64, 1])
Rango y en batch (log): [3.26, 5.69]


### Construcción RRNN

In [8]:
import torch.nn as nn


# =============================================================================
# ARQUITECTURA DE LA RED NEURONAL — AirbnbMLP
# =============================================================================
# MLP = Multi-Layer Perceptron (Perceptrón Multicapa)
# Es la arquitectura más simple de red neuronal densa: cada neurona de una
# capa está conectada con TODAS las neuronas de la capa siguiente.
#
# Para crear un modelo en PyTorch SIEMPRE hay que:
# 1. Heredar de nn.Module (la clase base de todos los modelos PyTorch)
# 2. Definir las capas en __init__
# 3. Definir el flujo de datos en forward()


class AirbnbMLP(nn.Module):


    def __init__(self, input_size):
        # Llamamos al constructor del padre (obligatorio en PyTorch)
        super(AirbnbMLP, self).__init__()


        # -----------------------------------------------------------------
        # DEFINICIÓN DE CAPAS
        # -----------------------------------------------------------------
        # nn.Sequential agrupa capas en orden. Los datos entran por la
        # primera y salen por la última, en secuencia automática.
        # Es más limpio que definir cada capa por separado.


        self.network = nn.Sequential(
            # CAPA 1: Capa de entrada → 128 neuronas
            nn.Linear(input_size, 128),  # capa densa con 128 neuronas
            nn.BatchNorm1d(128),         # Normaliza las activaciones entre capas
            nn.ReLU(),                   # Función de activación que introduce no-linealidad. Permite a la red aprender patrones complejos.
            nn.Dropout(p=0.2),           # Durante el entrenamiento, "apaga" aleatoriamente el 20% de las neuronas en cada forward pass.
                                         # Esto fuerza a la red a no depender de neuronas específicas y a aprender representaciones más robustas.
                                         # Resultado: menos overfitting (sobreajuste).
            # CAPA 2: 128 → 64 neuronas
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(p=0.2),
            # CAPA 3: 64 → 32 neuronas
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(p=0.2),
            # CAPA DE SALIDA: 32 → 1 neurona
            # Una sola neurona porque es un problema de REGRESIÓN:
            # queremos predecir UN número (el precio por noche).
            # Sin función de activación final → salida lineal ilimitada,
            # lo que permite predecir cualquier valor positivo o negativo.
            # (Si fuera clasificación usaríamos Softmax o Sigmoid aquí)
            nn.Linear(32, 1)
        )


    def forward(self, x):
        # forward() define cómo fluyen los datos a través de la red.
        # PyTorch llama a este método automáticamente cuando hacemos
        # predicciones: model(X_batch) → internamente ejecuta forward(X_batch)
        #
        # En nuestro caso es simple: los datos entran y recorren
        # todas las capas del Sequential en orden.
        return self.network(x)


# =============================================================================
# INSTANCIAR EL MODELO Y MOVERLO A LA GPU
# =============================================================================
# .to(device) mueve TODOS los parámetros del modelo (pesos y biases)
# a la GPU. A partir de aquí, los cálculos ocurren en la gráfica.
INPUT_SIZE = X_train_scaled.shape[1]
model_log = AirbnbMLP(input_size=INPUT_SIZE).to(device)


# =============================================================================
# INSPECCIÓN DEL MODELO
# =============================================================================
# print(model) muestra la arquitectura completa con todas las capas.
# Útil para verificar que la estructura es la esperada.
print(model_log)
print(f"\nDispositivo del modelo: {next(model_log.parameters()).device}")
print(f"INPUT_SIZE detectado automáticamente: {INPUT_SIZE}")

# Contamos el total de parámetros entrenables (pesos + biases)
# Es una métrica estándar para medir la "capacidad" de la red.
total_params = sum(p.numel() for p in model_log.parameters() if p.requires_grad)
print(f"Total parámetros entrenables: {total_params:,}")

AirbnbMLP(
  (network): Sequential(
    (0): Linear(in_features=73, out_features=128, bias=True)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.2, inplace=False)
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.2, inplace=False)
    (8): Linear(in_features=64, out_features=32, bias=True)
    (9): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): Dropout(p=0.2, inplace=False)
    (12): Linear(in_features=32, out_features=1, bias=True)
  )
)

Dispositivo del modelo: cuda:0
INPUT_SIZE detectado automáticamente: 73
Total parámetros entrenables: 20,289


### Entrenamiento

In [9]:
import torch.nn as nn
import torch.optim as optim
import copy
import numpy as np


# =============================================================================
# CONFIGURACIÓN DE HIPERPARÁMETROS
# =============================================================================
LEARNING_RATE = 0.001  # Tasa de aprendizaje del optimizador Adam
EPOCHS        = 5000   # Número máximo de épocas (el Early Stopping parará antes)
PATIENCE      = 30     # Early Stopping: épocas sin mejora antes de parar


# =============================================================================
# FUNCIÓN DE PÉRDIDA Y OPTIMIZADOR
# =============================================================================
# MSELoss (Mean Squared Error): penaliza los errores grandes de forma cuadrática.
# Si el modelo falla por 10€, el error es 100. Si falla por 50€, el error es 2500.
# Esto obliga a la red a evitar errores graves, ideal para precios de Airbnb.
# NOTA: con log1p el MSE opera en espacio logarítmico (valores pequeños ~0.01-0.5)
# pero el MAE, RMSE y R² se convierten de vuelta a euros reales con expm1.
criterion = nn.MSELoss()

# Adam (Adaptive Moment Estimation): optimizador que ajusta automáticamente
# la tasa de aprendizaje por parámetro. Es el estándar en Deep Learning.
optimizer_log = optim.Adam(model_log.parameters(), lr=LEARNING_RATE)


# =============================================================================
# LÓGICA DE EARLY STOPPING
# =============================================================================
# Variables de control para detener el entrenamiento si el modelo
# deja de mejorar en el conjunto de validación.
mejor_val_loss_log     = float('inf')  # Inicializamos con infinito para que cualquier pérdida real sea mejor en la primera época
epocas_sin_mejora_log  = 0             # Contador de épocas consecutivas sin mejora
mejor_modelo_pesos_log = None          # Guardaremos aquí los pesos del mejor modelo
mejor_r2_log           = None          # R² del mejor modelo, para el resumen final


# =============================================================================
# LISTAS PARA GUARDAR EL HISTORIAL DE ENTRENAMIENTO
# =============================================================================
# Las usaremos después para graficar y comparar con el modelo sin log
historial_train_loss_log = []
historial_train_mae_log  = []
historial_val_loss_log   = []
historial_val_mae_log    = []   # En euros reales (tras expm1)
historial_val_rmse_log   = []   # En euros reales (tras expm1)
historial_r2_log         = []   # En euros reales (tras expm1)


# =============================================================================
# BUCLE DE ENTRENAMIENTO PRINCIPAL
# =============================================================================
print(f"Iniciando entrenamiento (con log1p) en: {device}")
print(f"{'Época':<8} {'Train MSE':>10} {'Train MAE':>10} {'Val MSE':>10} {'Val MAE€':>10} {'Val RMSE€':>11} {'R²':>8} {'Estado':>10}")
print("-" * 95)


for epoca in range(1, EPOCHS + 1):


    # -------------------------------------------------------------------------
    # FASE 1: ENTRENAMIENTO (modo train)
    # -------------------------------------------------------------------------
    # model.train() activa el modo entrenamiento:
    # - Dropout está ACTIVO (apaga neuronas aleatoriamente)
    # - Los gradientes se calculan y acumulan
    model_log.train()
    train_loss_acumulado = 0.0
    train_mae_acumulado  = 0.0


    for X_batch, y_batch in loader_train:

        # Mover los tensores del batch a la GPU en cada iteración.
        # Los datos viven en RAM (CPU) y los movemos a VRAM (GPU) por lotes.
        # No podemos moverlos todos de golpe porque la VRAM tiene límite.
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        # PASO 1: Borrar gradientes acumulados del batch anterior.
        # PyTorch ACUMULA gradientes por defecto, si no los borramos
        # se sumarían a los del batch anterior y los pesos se actualizarían
        # con información incorrecta.
        optimizer_log.zero_grad()

        # PASO 2: Forward pass — los datos fluyen a través de la red
        # y obtenemos las predicciones del batch actual.
        predicciones = model_log(X_batch)

        # PASO 3: Calcular el error (Loss) entre predicciones y valores reales.
        # En espacio logarítmico: valores pequeños y estables (~0.01 - 0.5)
        loss = criterion(predicciones, y_batch)

        # PASO 4: Backward pass — PyTorch calcula automáticamente el gradiente
        # de la pérdida respecto a CADA parámetro de la red (regla de la cadena).
        # Esto indica en qué dirección y cuánto hay que mover cada peso.
        loss.backward()

        # PASO 5: El optimizador actualiza los pesos usando los gradientes
        # calculados en el backward pass.
        optimizer_log.step()

        # Acumulamos la pérdida para calcular el promedio de la época.
        # .item() extrae el valor escalar del tensor (necesario para operar en CPU)
        train_loss_acumulado += loss.item()

        # Train MAE en espacio log: sirve para comparar con Val MAE log
        # y detectar overfitting (si train baja pero val sube, hay sobreajuste)
        train_mae_acumulado += torch.mean(torch.abs(predicciones - y_batch)).item()

    # Pérdida media de todos los batches de entrenamiento en esta época
    train_loss_media = train_loss_acumulado / len(loader_train)
    train_mae_media  = train_mae_acumulado  / len(loader_train)


    # -------------------------------------------------------------------------
    # FASE 2: VALIDACIÓN (modo eval)
    # -------------------------------------------------------------------------
    # model.eval() activa el modo evaluación:
    # - Dropout está DESACTIVADO (todas las neuronas activas → predicciones estables)
    # - BatchNorm usa estadísticas fijas
    model_log.eval()
    val_loss_acumulado = 0.0
    val_mae_acumulado  = 0.0
    val_rmse_acumulado = 0.0

    # Para R² necesitamos acumular TODAS las predicciones y reales del epoch completo
    # No podemos calcularlo batch a batch porque R² necesita la media global
    todas_preds  = []
    todos_reales = []

    # torch.no_grad() desactiva el cálculo de gradientes durante la validación.
    # No necesitamos gradientes porque no vamos a actualizar pesos.
    # Ahorra memoria en GPU y acelera el cálculo ~2x.
    with torch.no_grad():
        for X_batch, y_batch in loader_val:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            predicciones = model_log(X_batch)

            # MSE Loss en espacio log → usado para el criterio de Early Stopping
            val_loss_acumulado += criterion(predicciones, y_batch).item()

            # expm1 es la inversa de log1p: expm1(log1p(x)) = x
            # Convierte las predicciones logarítmicas de vuelta a euros reales
            # para que MAE, RMSE y R² sean interpretables por humanos
            preds_euros  = torch.expm1(predicciones)
            reales_euros = torch.expm1(y_batch)

            # MAE (Mean Absolute Error) en euros reales:
            # Si MAE = 25 significa que el modelo falla de media 25€/noche.
            # Mucho más intuitivo que el MSE para comunicar resultados.
            val_mae_acumulado += torch.mean(
                torch.abs(preds_euros - reales_euros)
            ).item()

            # RMSE (Root Mean Squared Error) en euros reales:
            # Similar al MAE pero penaliza más los errores grandes.
            # Útil para detectar si el modelo falla mucho en precios extremos.
            val_rmse_acumulado += torch.sqrt(
                torch.mean((preds_euros - reales_euros) ** 2)
            ).item()

            # Acumulamos predicciones y reales (en euros) para calcular R²
            todas_preds.extend(preds_euros.cpu().numpy().flatten())
            todos_reales.extend(reales_euros.cpu().numpy().flatten())

    val_loss_media = val_loss_acumulado / len(loader_val)
    val_mae_media  = val_mae_acumulado  / len(loader_val)
    val_rmse_media = val_rmse_acumulado / len(loader_val)

    # R² (Coeficiente de Determinación) en euros reales:
    # Mide qué porcentaje de la variación del precio explica el modelo.
    # R² = 1 - (suma errores cuadráticos / varianza total del target)
    # R²=1.0 → predicción perfecta
    # R²=0.0 → el modelo no es mejor que predecir siempre la media
    # R²<0.0 → el modelo es peor que predecir siempre la media
    todas_preds  = np.array(todas_preds)
    todos_reales = np.array(todos_reales)
    ss_res = np.sum((todos_reales - todas_preds) ** 2)
    ss_tot = np.sum((todos_reales - np.mean(todos_reales)) ** 2)
    r2 = 1 - (ss_res / ss_tot)

    # Guardamos en el historial para las gráficas y comparación posterior
    historial_train_loss_log.append(train_loss_media)
    historial_train_mae_log.append(train_mae_media)
    historial_val_loss_log.append(val_loss_media)
    historial_val_mae_log.append(val_mae_media)
    historial_val_rmse_log.append(val_rmse_media)
    historial_r2_log.append(r2)


    # -------------------------------------------------------------------------
    # LÓGICA DE EARLY STOPPING
    # -------------------------------------------------------------------------
    if val_loss_media < mejor_val_loss_log:
        # El modelo mejoró: actualizamos el mejor loss, reseteamos el contador
        # y guardamos una copia de los pesos actuales en memoria.
        # .state_dict() devuelve un diccionario con todos los pesos y biases.
        # copy.deepcopy garantiza que guardamos una copia independiente,
        # no una referencia que se sobreescribiría en la siguiente época.
        mejor_val_loss_log     = val_loss_media
        epocas_sin_mejora_log  = 0
        mejor_modelo_pesos_log = copy.deepcopy(model_log.state_dict())
        mejor_r2_log           = r2
        estado = "✅ Mejor"
    else:
        epocas_sin_mejora_log += 1
        estado = f"⏳ {epocas_sin_mejora_log}/{PATIENCE}"

    # Imprimir progreso cada 5 épocas (o en la primera)
    if epoca % 5 == 0 or epoca == 1:
        print(f"{epoca:<8} {train_loss_media:>10.4f} {train_mae_media:>10.4f} {val_loss_media:>10.4f} {val_mae_media:>10.2f} {val_rmse_media:>11.2f} {r2:>8.4f} {estado:>10}")

    # Condición de parada: si llevamos PATIENCE épocas sin mejorar, paramos
    if epocas_sin_mejora_log >= PATIENCE:
        print(f"\n⛔ Early Stopping activado en época {epoca}.")
        print(f"   El modelo no mejoró en {PATIENCE} épocas consecutivas.")
        break


# =============================================================================
# RESTAURAR LOS MEJORES PESOS Y GUARDAR EL MODELO
# =============================================================================
# Cargamos los pesos de la época donde el modelo tuvo menor val_loss.
# Sin esto, el modelo final tendría los pesos de la última época,
# que podrían estar sobreajustados.
model_log.load_state_dict(mejor_modelo_pesos_log)

# Guardamos el modelo entrenado a disco para usarlo en la API.
# state_dict() es preferible a guardar el modelo entero porque es
# más portable y no depende de la estructura de clases del proyecto.
torch.save(mejor_modelo_pesos_log, '../data/airbnb_mlp_Log.pt')

print(f"\n{'='*60}")
print(f"✅ Modelo guardado en '../data/airbnb_mlp_Log.pt'")
print(f"   Mejor Val MSE (log): {mejor_val_loss_log:.4f}")
print(f"   Mejor Val MAE:       {min(historial_val_mae_log):.2f} €/noche")
print(f"   Mejor Val RMSE:      {min(historial_val_rmse_log):.2f} €/noche")
print(f"   Mejor R²:            {mejor_r2_log:.4f}  →  El modelo explica el {mejor_r2_log*100:.1f}% de la variación de precios")

Iniciando entrenamiento (con log1p) en: cuda
Época     Train MSE  Train MAE    Val MSE   Val MAE€   Val RMSE€       R²     Estado
-----------------------------------------------------------------------------------------------
1           14.0197     3.6829    12.0491     116.35      153.95  -0.8643    ✅ Mejor
5            0.8075     0.7128     0.3610      50.35       90.83   0.0598    ✅ Mejor
10           0.6206     0.6157     0.1647      37.47       76.25   0.2026    ✅ Mejor
15           0.5193     0.5693     0.1436      35.67       74.82   0.2020     ⏳ 1/30
20           0.4788     0.5407     0.1329      34.16       73.78   0.2085     ⏳ 6/30
25           0.4593     0.5319     0.1245      33.77       71.92   0.2323     ⏳ 2/30
30           0.3997     0.5015     0.1348      35.53       75.67   0.1951     ⏳ 2/30
35           0.3924     0.4944     0.1385      35.94       75.39   0.2015     ⏳ 7/30
40           0.3864     0.4908     0.1233      33.81       72.95   0.2213     ⏳ 1/30
45       

### Análisis RRNN V2 transformación logarítmica

El modelo sigue arrojando unos resultados débiles con apenas un 29% de accuracy y un MAE de 28-30€/noche. Por ello para mejorar el modelo se nos presentan las siguientes opciones:

| Solución                                   | Esfuerzo | `Posible Impacto esperado |
| ------------------------------------------ | -------- | --------------------------|
| Avanzar a Fase 3 (añadir fotos)            | Alto     | +15-25% R²                |
| Limpiar outliers del target (> 600€)       | Bajo     | +5-10% R²                 |
| Eliminar features de bajo poder predictivo | Medio    | +3-7% R²                  |
| Ajustar hiperparámetros                    | Medio    | +2-5% R²                  |

In [10]:
df_limpio.to_csv('../data/listingV6.csv', index=False)

## RRNN V3

#### Activar Gráfica

In [11]:
# pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121
import torch

print(torch.__version__)               # Debe contener "cu121" o similar, NO "cpu"
print(torch.cuda.is_available())       # Debe imprimir: True
print(torch.cuda.get_device_name(0))   # Debe imprimir el nombre de tu GPU, ej: "NVIDIA GeForce RTX 3060"
device = (
    "cuda"  if torch.cuda.is_available() else
    "mps"   if torch.backends.mps.is_available() else  # Apple Silicon
    "cpu"
)
print(f"Usando dispositivo: {device}")
# Output esperado en tu caso → "Usando dispositivo: cuda"

2.5.1
True
NVIDIA GeForce RTX 4070 Laptop GPU
Usando dispositivo: cuda


En el entrenamiento de esta RRNN se tomarán las siguientes medidas para intentar mejorar la precisión del modelo: Limpieza de outliers, limpieza de variables que provocan ruido y pruebas de ajuste de hiperparámetros.

### Limpieza outliers

En el EDA se llegó a la conclusión que los apartamentos por encima de 500-600€ son apartamentos de lujo o con precios desorbitados, los verdaderos prosefionales del sector se dedican al alquiler turístico de gran rotación que suele darse mediante apartmentos compartidos, habitaciones privadas o pisos de 1 o 2 habitaciones.

#### Estadísticas precios

In [12]:
import pandas as pd

df = pd.read_csv('../data/listingV5.csv') # Carga de datos

# Seaprar features y target
X = df.drop(columns=['price'])
y = df['price']

precios = pd.Series(y)

# --- Estadísticas descriptivas ---
print(precios.describe(percentiles=[.25, .50, .75, .90, .95, .99]))
print(f"\nPisos > 300€: {(precios > 300).sum()} ({(precios > 300).mean()*100:.1f}%)")
print(f"Pisos > 400€: {(precios > 400).sum()} ({(precios > 400).mean()*100:.1f}%)")
print(f"Pisos > 500€: {(precios > 500).sum()} ({(precios > 500).mean()*100:.1f}%)")
print(f"Pisos > 600€: {(precios > 600).sum()} ({(precios > 600).mean()*100:.1f}%)")

count    5796.000000
mean      118.566425
std       100.556774
min        18.000000
25%        75.000000
50%        98.000000
75%       134.000000
90%       189.000000
95%       246.000000
99%       450.000000
max      2932.000000
Name: price, dtype: float64

Pisos > 300€: 174 (3.0%)
Pisos > 400€: 84 (1.4%)
Pisos > 500€: 44 (0.8%)
Pisos > 600€: 29 (0.5%)


Con una media de 99€ y un máximo de 2932€, esos 44 pisos por encima de 500€ (0.8%) están distorsionando enormemente el entrenamiento.

In [13]:
PRICE_CAP = 450  # Umbral en €/noche — probar también con 450 si los resultados no mejoran

mascara = y <= PRICE_CAP
X_filtrado = X[mascara].copy()
y_filtrado = y[mascara].copy()

print(f"Apartamentos originales:  {len(y)}")
print(f"Apartamentos eliminados:  {(~mascara).sum()} (precio > {PRICE_CAP}€)")
print(f"Apartamentos restantes:   {len(y_filtrado)}")
print(f"\nNueva distribución del precio:")
print(y_filtrado.describe(percentiles=[.25, .50, .75, .90, .95, .99]))

Apartamentos originales:  5796
Apartamentos eliminados:  57 (precio > 450€)
Apartamentos restantes:   5739

Nueva distribución del precio:
count    5739.000000
mean      112.034501
std        61.915302
min        18.000000
25%        75.000000
50%        98.000000
75%       132.000000
90%       184.200000
95%       231.000000
99%       363.860000
max       450.000000
Name: price, dtype: float64


In [14]:
# Filtrar directamente sobre el dataframe limpio y guardar
df_filtrado = df[df['price'] <= 500].copy()

print(f"Apartamentos originales: {len(df)}")
print(f"Apartamentos restantes:  {len(df_filtrado)}")
print(f"Eliminados:              {len(df) - len(df_filtrado)}")

Apartamentos originales: 5796
Apartamentos restantes:  5752
Eliminados:              44


#### Matriz correlación

In [15]:
#Correlacion de las columnas con el precio por noche
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import plotly.graph_objects as go

correlacion = df_filtrado.corr(numeric_only=True)['price'].drop('price').sort_values()

colors = ['#ef4444' if v < 0 else '#3b82f6' for v in correlacion.values]

fig = go.Figure(go.Bar(
    x=correlacion.values,
    y=correlacion.index,
    orientation='h',
    marker_color=colors,
    text=[f"{v:.3f}" for v in correlacion.values],
    textposition='outside'
))

fig.update_layout(title="Correlación variables numéricas con Price", height=1400)
fig.update_xaxes(title_text="Correlación r")
fig.update_yaxes(title_text="Variable")
fig.add_vline(x=0.1, line_dash="dash", line_color="gray", opacity=0.5)
fig.add_vline(x=-0.1, line_dash="dash", line_color="gray", opacity=0.5)
fig.show()

# Guardar como imagen
#fig.write_image('../data/correlacion_precio.png', width=1400, height=1400, scale=2)
print("Imagen guardada en ../data/correlacion_precio.png")

Imagen guardada en ../data/correlacion_precio.png


In [16]:
from scipy import stats
import pandas as pd

cols_categoricas = df_filtrado.select_dtypes(include='object').columns.tolist()

resultados_anova = []

for col in cols_categoricas:
    grupos = [df_filtrado[df_filtrado[col] == cat]['price'].dropna()
            for cat in df_filtrado[col].dropna().unique()
            if len(df_filtrado[df_filtrado[col] == cat]['price'].dropna()) > 1]  # evitar grupos vacíos
    
    if len(grupos) > 1:  # ANOVA necesita al menos 2 grupos
        f_stat, p_valor = stats.f_oneway(*grupos)
        resultados_anova.append({'columna': col, 'p_valor': round(p_valor, 6)})
    else:
        resultados_anova.append({'columna': col, 'p_valor': None})

resultados_anova_df = pd.DataFrame(resultados_anova).sort_values('p_valor')
print(resultados_anova_df.to_string())



                  columna   p_valor
1  neighbourhood_cleansed  0.000000
2           property_type  0.000000
3               room_type  0.000000
0      host_response_time  0.270191


In [17]:
columnas_ruidosas = ['property_type', 'neighbourhood_cleansed', 'number_of_reviews', 'minimum_nights', 'latitude', 'host_responserate', 'review_scores_checkin', 'instant_bookable',
                    'maximum_nights', 'availability_365', 'review_scores_communication', 'has_refrigerator', 'has_microwave', 'has_hair_dryer', 'has_bed_linens', 'has_kitchen', 'has_iron',
                    'host_listings_counts', 'has_toaster']
df_filtrado = df_filtrado.drop(columns=columnas_ruidosas, errors='ignore')

columnas_leakage = ['id', 'price_per_person', 'estimated_occupancy_l365d', 'estimated_revenue_l365d']
df_filtrado = df_filtrado.drop(columns=columnas_leakage, errors='ignore')

In [18]:
df_filtrado.to_csv('../data/listingV5priceCAP.csv', index=False)

### Construcción RRNN

In [19]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
import joblib

df = pd.read_csv('../data/listingV5priceCAP.csv') # Carga de datos

# Seaprar features y target
X = df.drop(columns=['price'])
y = df['price']

#separar columnas por tipo
categorical_cols = ['host_response_time','room_type']
numerical_cols   = [col for col in X.columns if col not in categorical_cols]

print(f"\nColumnas numéricas ({len(numerical_cols)}): {numerical_cols}")

#ColumnTransformer
numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),  # Rellena NaN con la mediana del Train
    ('scaler',  StandardScaler())                   # Escala a media=0, desviación=1
])

# --- Pipeline para variables categóricas ---
# Añadimos también imputación categórica por si hubiera NaN en texto
# strategy='most_frequent' rellena con la categoría más común
categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# --- ColumnTransformer actualizado ---
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_pipeline,      numerical_cols),
        ('cat', categorical_pipeline,  categorical_cols)
    ],
    remainder='drop'
)

# División dataset
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42   # random_state=42 → reproducibilidad
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.20, random_state=42
)



# FIT TRANSFORM
X_train_scaled = preprocessor.fit_transform(X_train)
X_val_scaled   = preprocessor.transform(X_val)
X_test_scaled  = preprocessor.transform(X_test)


# =============================================================================
# PASO 9: CONVERTIR TARGETS A FLOAT32
# =============================================================================
# PyTorch trabaja internamente con tensores de tipo float32 (32 bits).
# Pandas usa float64 (64 bits) por defecto.
# Convertimos ahora para evitar errores de tipo en el DataLoader de PyTorch.
y_train = y_train.to_numpy(dtype=np.float32)
y_val   = y_val.to_numpy(dtype=np.float32)
y_test  = y_test.to_numpy(dtype=np.float32)

y_train = np.log1p(y_train).astype(np.float32)
y_val  = np.log1p(y_val).astype(np.float32)
# y_test lo guardamos en euros reales para la evaluación final
y_test = y_test.copy()  # copia sin transformar para métricas finales

print(f"Rango precios log-escalados:  [{y_train.min():.2f}, {y_train.max():.2f}]")
print(f"Media log-escalada:           {y_train.mean():.2f}")


# =============================================================================
# PASO 10: GUARDAR EL PREPROCESSOR PARA PRODUCCIÓN
# =============================================================================

#Por qué no guardar en CSV entonces
#Un CSV solo guarda los datos ya transformados, pero no guarda la receta de cómo transformarlos.

# Con joblib.dump() serializamos el objeto preprocessor a disco.
# Cuando despleguemos la API (FastAPI/Flask), haremos joblib.load() para
# recuperarlo y aplicar exactamente la misma transformación a los datos
# que lleguen de los usuarios, garantizando consistencia total.
import os
os.makedirs('../data', exist_ok=True)
joblib.dump(preprocessor, '../data/preprocessor.pkl')


# =============================================================================
# RESUMEN FINAL
# =============================================================================
print(f"\n✅ Train:      {X_train_scaled.shape}")
print(f"✅ Validation: {X_val_scaled.shape}")
print(f"✅ Test:       {X_test_scaled.shape}")
print(f"\n🧠 Variables de entrada al modelo: {X_train_scaled.shape[1]}")
print("¡Datos listos para PyTorch!")





Columnas numéricas (29): ['host_response_rate', 'host_is_superhost', 'host_listings_count', 'longitude', 'accommodates', 'bathrooms', 'bedrooms', 'beds', 'number_of_reviews_ltm', 'review_scores_rating', 'review_scores_accuracy', 'review_scores_cleanliness', 'review_scores_location', 'review_scores_value', 'reviews_per_month', 'private_bathroom', 'has_cooking_basics', 'has_tv', 'has_air_conditioning', 'has_washer', 'has_heating', 'has_freezer', 'has_coffee_maker', 'has_balcony_or_terrace', 'distancia_centro_km', 'personas_por_habitacion', 'banos_por_huesped', 'amenities_score', 'distancia_playa_km']
Rango precios log-escalados:  [3.00, 6.19]
Media log-escalada:           4.60

✅ Train:      (3680, 37)
✅ Validation: (921, 37)
✅ Test:       (1151, 37)

🧠 Variables de entrada al modelo: 37
¡Datos listos para PyTorch!


#### Clase AirbnbDataset

In [20]:
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np

# =============================================================================
# CLASE AirbnbDataset — Envuelve nuestros arrays NumPy en formato PyTorch
# =============================================================================
# Para crear un Dataset personalizado en PyTorch SIEMPRE hay que:
# 1. Heredar de la clase base 'Dataset'
# 2. Implementar obligatoriamente estos tres métodos:
#    - __init__  → constructor, guarda los datos
#    - __len__   → cuántos ejemplos tiene el dataset
#    - __getitem__→ cómo obtener el ejemplo número 'idx'
#
# El DataLoader llamará a __getitem__ e __len__ automáticamente. Nunca debemos llamarlo directamente

class AirbnbDataset(Dataset):

    def __init__(self, X, y):
        # Convertimos los arrays NumPy a tensores de PyTorch.
        # float32 es el tipo estándar para redes neuronales (float64 consume
        # el doble de memoria en GPU sin aportar precisión relevante).
        #
        # X ya es un array NumPy (salida del ColumnTransformer)
        # y ya es un array NumPy float32 (lo convertimos antes con to_numpy)
        self.X = torch.tensor(X, dtype=torch.float32)
        
        # .unsqueeze(1) convierte y de forma [N] a forma [N, 1]
        # Ejemplo: [120.0, 85.0, 200.0] → [[120.0], [85.0], [200.0]]
        # Esto es necesario porque la última capa de la red también
        # devuelve forma [N, 1], y PyTorch exige que ambos tengan
        # la misma forma para calcular el error (MSELoss).
        self.y = torch.tensor(y, dtype=torch.float32).unsqueeze(1)

    def __len__(self):
        # Devuelve el número total de ejemplos del dataset.
        # El DataLoader lo usa para saber cuántos batches hay por época.
        return len(self.X)

    def __getitem__(self, idx):
        # Dado un índice, devuelve el par (features, precio) correspondiente.
        # El DataLoader llama a este método batch_size veces por cada lote.
        return self.X[idx], self.y[idx]


# =============================================================================
# CREAR LOS TRES DATASETS
# =============================================================================
# Simplemente instanciamos la clase con cada split que preparamos antes.

dataset_train = AirbnbDataset(X_train_scaled, y_train)
dataset_val   = AirbnbDataset(X_val_scaled,   y_val)
dataset_test  = AirbnbDataset(X_test_scaled,  y_test)


# =============================================================================
# CREAR LOS TRES DATALOADERS
# =============================================================================
# El DataLoader es el que realmente alimenta la red durante el entrenamiento.
#
# Parámetros clave:
#
# · batch_size=64:
#   En lugar de pasar los ~4000 apartamentos de golpe (lo que saturaría
#   la memoria y daría un gradiente muy "promediado"), los pasamos de 64
#   en 64. Cada lote actualiza los pesos una vez.
#   64 es un buen balance entre velocidad y estabilidad del gradiente.
#
# · shuffle=True (solo en Train):
#   En cada época, baraja el orden de los apartamentos antes de formar
#   los batches. Esto evita que la red memorice el orden de los datos
#   y mejora la generalización.
#   En Validation y Test NO se baraja porque solo estamos evaluando,
#   el orden no afecta al resultado.
#
# · num_workers=0:
#   Número de procesos paralelos para cargar datos. En Windows, usar
#   num_workers > 0 dentro de un notebook causa errores de multiproceso,
#   así que lo dejamos en 0 (carga en el proceso principal).
#   En Linux/Mac se puede subir a 2 o 4 para más velocidad.

BATCH_SIZE = 64

loader_train = DataLoader(dataset_train, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
loader_val   = DataLoader(dataset_val,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
loader_test  = DataLoader(dataset_test,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)


# =============================================================================
# VERIFICACIÓN — Comprobamos que todo tiene la forma correcta
# =============================================================================
# Extraemos un batch de prueba del loader de train para inspeccionarlo.
X_batch_ejemplo, y_batch_ejemplo = next(iter(loader_train))

print(f"Tamaño del dataset Train:      {len(dataset_train)} apartamentos")
print(f"Tamaño del dataset Validation: {len(dataset_val)} apartamentos")
print(f"Tamaño del dataset Test:       {len(dataset_test)} apartamentos")
print(f"\nForma de un batch de X: {X_batch_ejemplo.shape}")  # [64, num_features]
print(f"Forma de un batch de y: {y_batch_ejemplo.shape}")  # [64, 1]
print(f"\nNúmero de batches por época (train): {len(loader_train)}")
print(f"Tipo de tensor X: {X_batch_ejemplo.dtype}")
print(f"Tipo de tensor y: {y_batch_ejemplo.dtype}")

Tamaño del dataset Train:      3680 apartamentos
Tamaño del dataset Validation: 921 apartamentos
Tamaño del dataset Test:       1151 apartamentos

Forma de un batch de X: torch.Size([64, 37])
Forma de un batch de y: torch.Size([64, 1])

Número de batches por época (train): 58
Tipo de tensor X: torch.float32
Tipo de tensor y: torch.float32


In [21]:
import torch.nn as nn


# =============================================================================
# ARQUITECTURA DE LA RED NEURONAL — AirbnbMLP
# =============================================================================
# MLP = Multi-Layer Perceptron (Perceptrón Multicapa)
# Es la arquitectura más simple de red neuronal densa: cada neurona de una
# capa está conectada con TODAS las neuronas de la capa siguiente.
#
# Para crear un modelo en PyTorch SIEMPRE hay que:
# 1. Heredar de nn.Module (la clase base de todos los modelos PyTorch)
# 2. Definir las capas en __init__
# 3. Definir el flujo de datos en forward()


class AirbnbMLP(nn.Module):


    def __init__(self, input_size):
        # Llamamos al constructor del padre (obligatorio en PyTorch)
        super(AirbnbMLP, self).__init__()


        # -----------------------------------------------------------------
        # DEFINICIÓN DE CAPAS
        # -----------------------------------------------------------------
        # nn.Sequential agrupa capas en orden. Los datos entran por la
        # primera y salen por la última, en secuencia automática.
        # Es más limpio que definir cada capa por separado.


        self.network = nn.Sequential(
            # CAPA 1: Capa de entrada → 128 neuronas
            nn.Linear(input_size, 128),  # capa densa con 128 neuronas
            nn.BatchNorm1d(128),         # Normaliza las activaciones entre capas
            nn.ReLU(),                   # Función de activación que introduce no-linealidad. Permite a la red aprender patrones complejos.
            nn.Dropout(p=0.2),           # Durante el entrenamiento, "apaga" aleatoriamente el 20% de las neuronas en cada forward pass.
                                         # Esto fuerza a la red a no depender de neuronas específicas y a aprender representaciones más robustas.
                                         # Resultado: menos overfitting (sobreajuste).
            # CAPA 2: 128 → 64 neuronas
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(p=0.2),
            # CAPA 3: 64 → 32 neuronas
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(p=0.2),
            # CAPA DE SALIDA: 32 → 1 neurona
            # Una sola neurona porque es un problema de REGRESIÓN:
            # queremos predecir UN número (el precio por noche).
            # Sin función de activación final → salida lineal ilimitada,
            # lo que permite predecir cualquier valor positivo o negativo.
            # (Si fuera clasificación usaríamos Softmax o Sigmoid aquí)
            nn.Linear(32, 1)
        )


    def forward(self, x):
        # forward() define cómo fluyen los datos a través de la red.
        # PyTorch llama a este método automáticamente cuando hacemos
        # predicciones: model(X_batch) → internamente ejecuta forward(X_batch)
        #
        # En nuestro caso es simple: los datos entran y recorren
        # todas las capas del Sequential en orden.
        return self.network(x)


# =============================================================================
# INSTANCIAR EL MODELO Y MOVERLO A LA GPU
# =============================================================================
# .to(device) mueve TODOS los parámetros del modelo (pesos y biases)
# a la GPU. A partir de aquí, los cálculos ocurren en la gráfica.
INPUT_SIZE = X_train_scaled.shape[1]
model = AirbnbMLP(input_size=INPUT_SIZE).to(device)


# =============================================================================
# INSPECCIÓN DEL MODELO
# =============================================================================
# print(model) muestra la arquitectura completa con todas las capas.
# Útil para verificar que la estructura es la esperada.
print(model)
print(f"\nDispositivo del modelo: {next(model.parameters()).device}")
print(f"INPUT_SIZE detectado automáticamente: {INPUT_SIZE}")

# Contamos el total de parámetros entrenables (pesos + biases)
# Es una métrica estándar para medir la "capacidad" de la red.
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parámetros entrenables: {total_params:,}")

AirbnbMLP(
  (network): Sequential(
    (0): Linear(in_features=37, out_features=128, bias=True)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.2, inplace=False)
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.2, inplace=False)
    (8): Linear(in_features=64, out_features=32, bias=True)
    (9): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): Dropout(p=0.2, inplace=False)
    (12): Linear(in_features=32, out_features=1, bias=True)
  )
)

Dispositivo del modelo: cuda:0
INPUT_SIZE detectado automáticamente: 37
Total parámetros entrenables: 15,681


#### Verificación precio en €

In [22]:
import numpy as np

y_batch_ejemplo = next(iter(loader_train))[1]
print(f"Rango del target en loader_train: [{y_batch_ejemplo.min():.2f}, {y_batch_ejemplo.max():.2f}]")
print(f"Media del target: {y_batch_ejemplo.mean():.2f}")

Rango del target en loader_train: [3.18, 6.11]
Media del target: 4.60


#### Entrenamiento

In [23]:
import torch.nn as nn
import torch.optim as optim
import copy
import numpy as np


# =============================================================================
# CONFIGURACIÓN DE HIPERPARÁMETROS
# =============================================================================
LEARNING_RATE = 0.001  # Tasa de aprendizaje del optimizador Adam
EPOCHS        = 5000   # Número máximo de épocas (el Early Stopping parará antes)
PATIENCE      = 30     # Early Stopping: épocas sin mejora antes de parar


# =============================================================================
# FUNCIÓN DE PÉRDIDA Y OPTIMIZADOR
# =============================================================================
# MSELoss (Mean Squared Error): penaliza los errores grandes de forma cuadrática.
# Si el modelo falla por 10€, el error es 100. Si falla por 50€, el error es 2500.
# Esto obliga a la red a evitar errores graves, ideal para precios de Airbnb.
criterion = nn.MSELoss()

# Adam (Adaptive Moment Estimation): optimizador que ajusta automáticamente
# la tasa de aprendizaje por parámetro. Es el estándar en Deep Learning.
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)


# =============================================================================
# LÓGICA DE EARLY STOPPING
# =============================================================================
# Variables de control para detener el entrenamiento si el modelo
# deja de mejorar en el conjunto de validación.
mejor_val_loss     = float('inf')  # Inicializamos con infinito para que cualquier pérdida real sea mejor en la primera época
epocas_sin_mejora  = 0             # Contador de épocas consecutivas sin mejora
mejor_modelo_pesos = None          # Guardaremos aquí los pesos del mejor modelo
mejor_r2           = None          # R² del mejor modelo, para el resumen final


# =============================================================================
# LISTAS PARA GUARDAR EL HISTORIAL DE ENTRENAMIENTO
# =============================================================================
# Las usaremos después para graficar las curvas de aprendizaje
historial_train_loss = []
historial_train_mae  = []
historial_val_loss   = []
historial_val_mae    = []
historial_val_rmse   = []
historial_r2         = []


# =============================================================================
# BUCLE DE ENTRENAMIENTO PRINCIPAL
# =============================================================================
print(f"Iniciando entrenamiento en: {device}")
print(f"{'Época':<8} {'Train MSE':>10} {'Train MAE':>10} {'Val MSE':>10} {'Val MAE':>10} {'Val RMSE':>10} {'R²':>8} {'Estado':>10}")
print("-" * 90)


for epoca in range(1, EPOCHS + 1):


    # -------------------------------------------------------------------------
    # FASE 1: ENTRENAMIENTO (modo train)
    # -------------------------------------------------------------------------
    # model.train() activa el modo entrenamiento:
    # - Dropout está ACTIVO (apaga neuronas aleatoriamente)
    # - Los gradientes se calculan y acumulan
    model.train()
    train_loss_acumulado = 0.0
    train_mae_acumulado  = 0.0


    for X_batch, y_batch in loader_train:

        # Mover los tensores del batch a la GPU en cada iteración.
        # Los datos viven en RAM (CPU) y los movemos a VRAM (GPU) por lotes.
        # No podemos moverlos todos de golpe porque la VRAM tiene límite.
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        # PASO 1: Borrar gradientes acumulados del batch anterior.
        # PyTorch ACUMULA gradientes por defecto, si no los borramos
        # se sumarían a los del batch anterior y los pesos se actualizarían
        # con información incorrecta.
        optimizer.zero_grad()

        # PASO 2: Forward pass — los datos fluyen a través de la red
        # y obtenemos las predicciones del batch actual.
        predicciones = model(X_batch)

        # PASO 3: Calcular el error (Loss) entre predicciones y valores reales.
        loss = criterion(predicciones, y_batch)

        # PASO 4: Backward pass — PyTorch calcula automáticamente el gradiente
        # de la pérdida respecto a CADA parámetro de la red (regla de la cadena).
        # Esto indica en qué dirección y cuánto hay que mover cada peso.
        loss.backward()

        # PASO 5: El optimizador actualiza los pesos usando los gradientes
        # calculados en el backward pass.
        optimizer.step()

        # Acumulamos la pérdida para calcular el promedio de la época.
        # .item() extrae el valor escalar del tensor (necesario para operar en CPU)
        train_loss_acumulado += loss.item()

        # MAE de entrenamiento: permite comparar visualmente con el MAE de validación
        # y detectar overfitting (si train MAE baja pero val MAE sube, hay sobreajuste)
        train_mae_acumulado += torch.mean(torch.abs(predicciones - y_batch)).item()

    # Pérdida media de todos los batches de entrenamiento en esta época
    train_loss_media = train_loss_acumulado / len(loader_train)
    train_mae_media  = train_mae_acumulado  / len(loader_train)


    # -------------------------------------------------------------------------
    # FASE 2: VALIDACIÓN (modo eval)
    # -------------------------------------------------------------------------
    # model.eval() activa el modo evaluación:
    # - Dropout está DESACTIVADO (todas las neuronas activas → predicciones estables)
    # - BatchNorm usa estadísticas fijas
    model.eval()
    val_loss_acumulado = 0.0
    val_mae_acumulado  = 0.0
    val_rmse_acumulado = 0.0

    # Para R² necesitamos acumular TODAS las predicciones y reales del epoch completo
    # No podemos calcularlo batch a batch porque R² necesita la media global
    todas_preds  = []
    todos_reales = []

    # torch.no_grad() desactiva el cálculo de gradientes durante la validación.
    # No necesitamos gradientes porque no vamos a actualizar pesos.
    # Ahorra memoria en GPU y acelera el cálculo ~2x.
    with torch.no_grad():
        for X_batch, y_batch in loader_val:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            predicciones = model(X_batch)

            # expm1 es la inversa exacta de log1p: convierte de vuelta a euros reales
            preds_euros  = torch.expm1(predicciones)
            reales_euros = torch.expm1(y_batch)
            
            # MSE Loss para el criterio de Early Stopping
            val_loss_acumulado += criterion(predicciones, y_batch).item()

            # MAE (Mean Absolute Error): fallo medio en €/noche
            # Mucho más intuitivo que el MSE para comunicar resultados.
            # Si MAE = 28 significa que el modelo falla de media 28€/noche.
            val_mae_acumulado  += torch.mean(torch.abs(preds_euros - reales_euros)).item()

            # RMSE (Root Mean Squared Error): raíz del error cuadrático medio.
            # Similar al MAE pero penaliza más los errores grandes.
            # Útil para detectar si el modelo falla mucho en precios extremos.
            val_rmse_acumulado += torch.sqrt(torch.mean((preds_euros - reales_euros) ** 2)).item()

            # Acumulamos predicciones y reales para calcular R² al final de la época
            # Para R², acumular también en euros reales
            todas_preds.extend(preds_euros.cpu().numpy().flatten())
            todos_reales.extend(reales_euros.cpu().numpy().flatten())


    val_loss_media = val_loss_acumulado / len(loader_val)
    val_mae_media  = val_mae_acumulado  / len(loader_val)
    val_rmse_media = val_rmse_acumulado / len(loader_val)

    # R² (Coeficiente de Determinación):
    # Mide qué porcentaje de la variación del precio explica el modelo.
    # R² = 1 - (suma errores cuadráticos / varianza total del target)
    # R²=1.0 → predicción perfecta
    # R²=0.0 → el modelo no es mejor que predecir siempre la media
    # R²<0.0 → el modelo es peor que predecir siempre la media
    todas_preds  = np.array(todas_preds)
    todos_reales = np.array(todos_reales)
    ss_res = np.sum((todos_reales - todas_preds) ** 2)
    ss_tot = np.sum((todos_reales - np.mean(todos_reales)) ** 2)
    r2 = 1 - (ss_res / ss_tot)

    # Guardamos en el historial para las gráficas posteriores
    historial_train_loss.append(train_loss_media)
    historial_train_mae.append(train_mae_media)
    historial_val_loss.append(val_loss_media)
    historial_val_mae.append(val_mae_media)
    historial_val_rmse.append(val_rmse_media)
    historial_r2.append(r2)


    # -------------------------------------------------------------------------
    # LÓGICA DE EARLY STOPPING
    # -------------------------------------------------------------------------
    if val_loss_media < mejor_val_loss:
        # El modelo mejoró: actualizamos el mejor loss, reseteamos el contador
        # y guardamos una copia de los pesos actuales en memoria.
        # .state_dict() devuelve un diccionario con todos los pesos y biases.
        # copy.deepcopy garantiza que guardamos una copia independiente,
        # no una referencia que se sobreescribiría en la siguiente época.
        mejor_val_loss     = val_loss_media
        epocas_sin_mejora  = 0
        mejor_modelo_pesos = copy.deepcopy(model.state_dict())
        mejor_r2           = r2
        estado = "✅ Mejor"
    else:
        epocas_sin_mejora += 1
        estado = f"⏳ {epocas_sin_mejora}/{PATIENCE}"

    # Imprimir progreso cada 5 épocas (o en la primera)
    if epoca % 5 == 0 or epoca == 1:
        print(f"{epoca:<8} {train_loss_media:>10.2f} {train_mae_media:>10.2f} {val_loss_media:>10.2f} {val_mae_media:>10.2f} {val_rmse_media:>10.2f} {r2:>8.4f} {estado:>10}")

    # Condición de parada: si llevamos PATIENCE épocas sin mejorar, paramos
    if epocas_sin_mejora >= PATIENCE:
        print(f"\n⛔ Early Stopping activado en época {epoca}.")
        print(f"   El modelo no mejoró en {PATIENCE} épocas consecutivas.")
        break


# =============================================================================
# RESTAURAR LOS MEJORES PESOS Y GUARDAR EL MODELO
# =============================================================================
# Cargamos los pesos de la época donde el modelo tuvo menor val_loss.
# Sin esto, el modelo final tendría los pesos de la última época,
# que podrían estar sobreajustados.
model.load_state_dict(mejor_modelo_pesos)

# Guardamos el modelo entrenado a disco para usarlo en la API.
# state_dict() es preferible a guardar el modelo entero porque es
# más portable y no depende de la estructura de clases del proyecto.
import os

MEJOR_MODELO_PATH = '../data/airbnb_mlp_PriceCAP.pt'

# Cargar el mejor R² histórico guardado en disco (si existe)
mejor_r2_historico = -float('inf')
if os.path.exists(MEJOR_MODELO_PATH.replace('.pt', '_meta.txt')):
    with open(MEJOR_MODELO_PATH.replace('.pt', '_meta.txt'), 'r') as f:
        mejor_r2_historico = float(f.read())

# Solo guardar si el nuevo modelo supera al histórico
if mejor_r2 > mejor_r2_historico:
    torch.save(mejor_modelo_pesos, MEJOR_MODELO_PATH)
    with open(MEJOR_MODELO_PATH.replace('.pt', '_meta.txt'), 'w') as f:
        f.write(str(mejor_r2))
    print(f"✅ Modelo guardado — nuevo R²: {mejor_r2:.4f} (anterior: {mejor_r2_historico:.4f})")
else:
    print(f"⏭️  Modelo NO guardado — R² actual {mejor_r2:.4f} no supera el histórico {mejor_r2_historico:.4f}")

print(f"\n{'='*60}")
#print(f"✅ Modelo guardado en '../data/airbnb_mlp_PriceCAP.pt'")
print(f"   Mejor Val MSE:  {mejor_val_loss:.2f}")
print(f"   Mejor Val MAE:  {min(historial_val_mae):.2f} €/noche")
print(f"   Mejor Val RMSE: {min(historial_val_rmse):.2f} €/noche")
print(f"   Mejor R²:       {mejor_r2:.4f}  →  El modelo explica el {mejor_r2*100:.1f}% de la variación de precios")

Iniciando entrenamiento en: cuda
Época     Train MSE  Train MAE    Val MSE    Val MAE   Val RMSE       R²     Estado
------------------------------------------------------------------------------------------
1             21.71       4.62      18.40     110.10     124.68  -3.3380    ✅ Mejor
5              1.52       1.03       0.80      62.15      78.41  -0.7523    ✅ Mejor
10             0.62       0.62       0.17      33.28      51.48   0.2240     ⏳ 1/30
15             0.55       0.58       0.14      31.86      48.25   0.3268    ✅ Mejor
20             0.50       0.56       0.14      31.06      49.34   0.2920     ⏳ 4/30
25             0.49       0.55       0.13      30.56      49.27   0.2959     ⏳ 2/30
30             0.44       0.53       0.13      30.63      47.14   0.3488     ⏳ 3/30
35             0.40       0.51       0.12      29.06      46.36   0.3670     ⏳ 3/30
40             0.39       0.49       0.13      29.80      48.03   0.3235     ⏳ 1/30
45             0.37       0.48      

### RRNN SIN TRANSFORMAR EL TARGET A LOGARITMO

#### Activar Gráfica

In [24]:
# pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121
import torch

print(torch.__version__)               # Debe contener "cu121" o similar, NO "cpu"
print(torch.cuda.is_available())       # Debe imprimir: True
print(torch.cuda.get_device_name(0))   # Debe imprimir el nombre de tu GPU, ej: "NVIDIA GeForce RTX 3060"
device = (
    "cuda"  if torch.cuda.is_available() else
    "mps"   if torch.backends.mps.is_available() else  # Apple Silicon
    "cpu"
)
print(f"Usando dispositivo: {device}")
# Output esperado en tu caso → "Usando dispositivo: cuda"

2.5.1
True
NVIDIA GeForce RTX 4070 Laptop GPU
Usando dispositivo: cuda


#### Construcción RRNN

In [25]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
import joblib

df = pd.read_csv('../data/listingV5priceCAP.csv')

X = df.drop(columns=['price'])
y = df['price']

categorical_cols = ['host_response_time', 'room_type']
numerical_cols   = [col for col in X.columns if col not in categorical_cols]

print(f"\nColumnas numéricas ({len(numerical_cols)}): {numerical_cols}")

numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])
categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_pipeline,     numerical_cols),
        ('cat', categorical_pipeline, categorical_cols)
    ],
    remainder='drop'
)

X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.20, random_state=42)

X_train_scaled = preprocessor.fit_transform(X_train)
X_val_scaled   = preprocessor.transform(X_val)
X_test_scaled  = preprocessor.transform(X_test)

# SIN log1p — target directamente en euros reales
y_train = y_train.to_numpy(dtype=np.float32)
y_val   = y_val.to_numpy(dtype=np.float32)
y_test  = y_test.to_numpy(dtype=np.float32)

print(f"Rango precios train: [{y_train.min():.0f}€, {y_train.max():.0f}€]")
print(f"Media precios train: {y_train.mean():.1f}€")

import os
os.makedirs('../data', exist_ok=True)
joblib.dump(preprocessor, '../data/preprocessor_raw.pkl')

print(f"\n✅ Train:      {X_train_scaled.shape}")
print(f"✅ Validation: {X_val_scaled.shape}")
print(f"✅ Test:       {X_test_scaled.shape}")
print(f"\n🧠 Variables de entrada al modelo: {X_train_scaled.shape[1]}")


Columnas numéricas (29): ['host_response_rate', 'host_is_superhost', 'host_listings_count', 'longitude', 'accommodates', 'bathrooms', 'bedrooms', 'beds', 'number_of_reviews_ltm', 'review_scores_rating', 'review_scores_accuracy', 'review_scores_cleanliness', 'review_scores_location', 'review_scores_value', 'reviews_per_month', 'private_bathroom', 'has_cooking_basics', 'has_tv', 'has_air_conditioning', 'has_washer', 'has_heating', 'has_freezer', 'has_coffee_maker', 'has_balcony_or_terrace', 'distancia_centro_km', 'personas_por_habitacion', 'banos_por_huesped', 'amenities_score', 'distancia_playa_km']
Rango precios train: [19€, 489€]
Media precios train: 112.8€

✅ Train:      (3680, 37)
✅ Validation: (921, 37)
✅ Test:       (1151, 37)

🧠 Variables de entrada al modelo: 37


#### Clase AirbnbDataset + DataLoaders

In [26]:
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np

# =============================================================================
# CLASE AirbnbDataset — Envuelve nuestros arrays NumPy en formato PyTorch
# =============================================================================
# Para crear un Dataset personalizado en PyTorch SIEMPRE hay que:
# 1. Heredar de la clase base 'Dataset'
# 2. Implementar obligatoriamente estos tres métodos:
#    - __init__  → constructor, guarda los datos
#    - __len__   → cuántos ejemplos tiene el dataset
#    - __getitem__→ cómo obtener el ejemplo número 'idx'
#
# El DataLoader llamará a __getitem__ e __len__ automáticamente. Nunca debemos llamarlo directamente

class AirbnbDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).unsqueeze(1)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


# =============================================================================
# CREAR LOS TRES DATASETS
# =============================================================================
# Simplemente instanciamos la clase con cada split que preparamos antes.

dataset_train = AirbnbDataset(X_train_scaled, y_train)
dataset_val   = AirbnbDataset(X_val_scaled,   y_val)
dataset_test  = AirbnbDataset(X_test_scaled,  y_test)


# =============================================================================
# CREAR LOS TRES DATALOADERS
# =============================================================================
# El DataLoader es el que realmente alimenta la red durante el entrenamiento.
#
# Parámetros clave:
#
# · batch_size=64:
#   En lugar de pasar los ~4000 apartamentos de golpe (lo que saturaría
#   la memoria y daría un gradiente muy "promediado"), los pasamos de 64
#   en 64. Cada lote actualiza los pesos una vez.
#   64 es un buen balance entre velocidad y estabilidad del gradiente.
#
# · shuffle=True (solo en Train):
#   En cada época, baraja el orden de los apartamentos antes de formar
#   los batches. Esto evita que la red memorice el orden de los datos
#   y mejora la generalización.
#   En Validation y Test NO se baraja porque solo estamos evaluando,
#   el orden no afecta al resultado.
#
# · num_workers=0:
#   Número de procesos paralelos para cargar datos. En Windows, usar
#   num_workers > 0 dentro de un notebook causa errores de multiproceso,
#   así que lo dejamos en 0 (carga en el proceso principal).
#   En Linux/Mac se puede subir a 2 o 4 para más velocidad.

BATCH_SIZE = 64
loader_train = DataLoader(dataset_train, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
loader_val   = DataLoader(dataset_val,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
loader_test  = DataLoader(dataset_test,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)


# =============================================================================
# VERIFICACIÓN — Comprobamos que todo tiene la forma correcta
# =============================================================================
# Extraemos un batch de prueba del loader de train para inspeccionarlo.
X_batch_ejemplo, y_batch_ejemplo = next(iter(loader_train))

print(f"Tamaño del dataset Train:      {len(dataset_train)} apartamentos")
print(f"Tamaño del dataset Validation: {len(dataset_val)} apartamentos")
print(f"Tamaño del dataset Test:       {len(dataset_test)} apartamentos")
print(f"\nForma de un batch de X: {X_batch_ejemplo.shape}")  # [64, num_features]
print(f"Forma de un batch de y: {y_batch_ejemplo.shape}")  # [64, 1]
print(f"\nNúmero de batches por época (train): {len(loader_train)}")
print(f"Tipo de tensor X: {X_batch_ejemplo.dtype}")
print(f"Tipo de tensor y: {y_batch_ejemplo.dtype}")

Tamaño del dataset Train:      3680 apartamentos
Tamaño del dataset Validation: 921 apartamentos
Tamaño del dataset Test:       1151 apartamentos

Forma de un batch de X: torch.Size([64, 37])
Forma de un batch de y: torch.Size([64, 1])

Número de batches por época (train): 58
Tipo de tensor X: torch.float32
Tipo de tensor y: torch.float32


In [27]:
import torch.nn as nn


# =============================================================================
# ARQUITECTURA DE LA RED NEURONAL — AirbnbMLP
# =============================================================================
# MLP = Multi-Layer Perceptron (Perceptrón Multicapa)
# Es la arquitectura más simple de red neuronal densa: cada neurona de una
# capa está conectada con TODAS las neuronas de la capa siguiente.
#
# Para crear un modelo en PyTorch SIEMPRE hay que:
# 1. Heredar de nn.Module (la clase base de todos los modelos PyTorch)
# 2. Definir las capas en __init__
# 3. Definir el flujo de datos en forward()


class AirbnbMLP(nn.Module):


    def __init__(self, input_size):
        # Llamamos al constructor del padre (obligatorio en PyTorch)
        super(AirbnbMLP, self).__init__()


        # -----------------------------------------------------------------
        # DEFINICIÓN DE CAPAS
        # -----------------------------------------------------------------
        # nn.Sequential agrupa capas en orden. Los datos entran por la
        # primera y salen por la última, en secuencia automática.
        # Es más limpio que definir cada capa por separado.


        self.network = nn.Sequential(
            # CAPA 1: Capa de entrada → 128 neuronas
            nn.Linear(input_size, 128),  # capa densa con 128 neuronas
            nn.BatchNorm1d(128),         # Normaliza las activaciones entre capas
            nn.ReLU(),                   # Función de activación que introduce no-linealidad. Permite a la red aprender patrones complejos.
            nn.Dropout(p=0.2),           # Durante el entrenamiento, "apaga" aleatoriamente el 20% de las neuronas en cada forward pass.
                                         # Esto fuerza a la red a no depender de neuronas específicas y a aprender representaciones más robustas.
                                         # Resultado: menos overfitting (sobreajuste).
            # CAPA 2: 128 → 64 neuronas
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(p=0.2),
            # CAPA 3: 64 → 32 neuronas
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(p=0.2),
            # CAPA DE SALIDA: 32 → 1 neurona
            # Una sola neurona porque es un problema de REGRESIÓN:
            # queremos predecir UN número (el precio por noche).
            # Sin función de activación final → salida lineal ilimitada,
            # lo que permite predecir cualquier valor positivo o negativo.
            # (Si fuera clasificación usaríamos Softmax o Sigmoid aquí)
            nn.Linear(32, 1)
        )


    def forward(self, x):
        # forward() define cómo fluyen los datos a través de la red.
        # PyTorch llama a este método automáticamente cuando hacemos
        # predicciones: model(X_batch) → internamente ejecuta forward(X_batch)
        #
        # En nuestro caso es simple: los datos entran y recorren
        # todas las capas del Sequential en orden.
        return self.network(x)


# =============================================================================
# INSTANCIAR EL MODELO Y MOVERLO A LA GPU
# =============================================================================
# .to(device) mueve TODOS los parámetros del modelo (pesos y biases)
# a la GPU. A partir de aquí, los cálculos ocurren en la gráfica.
INPUT_SIZE = X_train_scaled.shape[1]
model = AirbnbMLP(input_size=INPUT_SIZE).to(device)


# =============================================================================
# INSPECCIÓN DEL MODELO
# =============================================================================
# print(model) muestra la arquitectura completa con todas las capas.
# Útil para verificar que la estructura es la esperada.
print(model)
print(f"\nDispositivo del modelo: {next(model.parameters()).device}")
print(f"INPUT_SIZE detectado automáticamente: {INPUT_SIZE}")

# Contamos el total de parámetros entrenables (pesos + biases)
# Es una métrica estándar para medir la "capacidad" de la red.
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parámetros entrenables: {total_params:,}")

AirbnbMLP(
  (network): Sequential(
    (0): Linear(in_features=37, out_features=128, bias=True)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.2, inplace=False)
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.2, inplace=False)
    (8): Linear(in_features=64, out_features=32, bias=True)
    (9): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): Dropout(p=0.2, inplace=False)
    (12): Linear(in_features=32, out_features=1, bias=True)
  )
)

Dispositivo del modelo: cuda:0
INPUT_SIZE detectado automáticamente: 37
Total parámetros entrenables: 15,681


#### Arquitectura + entrenamiento

In [28]:
import torch.nn as nn
import torch.optim as optim
import copy
import numpy as np


# =============================================================================
# CONFIGURACIÓN DE HIPERPARÁMETROS
# =============================================================================
LEARNING_RATE = 0.001  # Tasa de aprendizaje del optimizador Adam
EPOCHS        = 5000   # Número máximo de épocas (el Early Stopping parará antes)
PATIENCE      = 30     # Early Stopping: épocas sin mejora antes de parar


# =============================================================================
# FUNCIÓN DE PÉRDIDA Y OPTIMIZADOR
# =============================================================================
# MSELoss (Mean Squared Error): penaliza los errores grandes de forma cuadrática.
# Si el modelo falla por 10€, el error es 100. Si falla por 50€, el error es 2500.
# Esto obliga a la red a evitar errores graves, ideal para precios de Airbnb.
criterion = nn.MSELoss()

# Adam (Adaptive Moment Estimation): optimizador que ajusta automáticamente
# la tasa de aprendizaje por parámetro. Es el estándar en Deep Learning.
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)


# =============================================================================
# LÓGICA DE EARLY STOPPING
# =============================================================================
# Variables de control para detener el entrenamiento si el modelo
# deja de mejorar en el conjunto de validación.
mejor_val_loss     = float('inf')  # Inicializamos con infinito para que cualquier pérdida real sea mejor en la primera época
epocas_sin_mejora  = 0             # Contador de épocas consecutivas sin mejora
mejor_modelo_pesos = None          # Guardaremos aquí los pesos del mejor modelo
mejor_r2           = None          # R² del mejor modelo, para el resumen final


# =============================================================================
# LISTAS PARA GUARDAR EL HISTORIAL DE ENTRENAMIENTO
# =============================================================================
# Las usaremos después para graficar las curvas de aprendizaje
historial_train_loss = []
historial_train_mae  = []
historial_val_loss   = []
historial_val_mae    = []
historial_val_rmse   = []
historial_r2         = []


# =============================================================================
# BUCLE DE ENTRENAMIENTO PRINCIPAL
# =============================================================================
print(f"Iniciando entrenamiento en: {device}")
print(f"{'Época':<8} {'Train MSE':>10} {'Train MAE':>10} {'Val MSE':>10} {'Val MAE':>10} {'Val RMSE':>10} {'R²':>8} {'Estado':>10}")
print("-" * 90)


for epoca in range(1, EPOCHS + 1):


    # -------------------------------------------------------------------------
    # FASE 1: ENTRENAMIENTO (modo train)
    # -------------------------------------------------------------------------
    # model.train() activa el modo entrenamiento:
    # - Dropout está ACTIVO (apaga neuronas aleatoriamente)
    # - Los gradientes se calculan y acumulan
    model.train()
    train_loss_acumulado = 0.0
    train_mae_acumulado  = 0.0


    for X_batch, y_batch in loader_train:

        # Mover los tensores del batch a la GPU en cada iteración.
        # Los datos viven en RAM (CPU) y los movemos a VRAM (GPU) por lotes.
        # No podemos moverlos todos de golpe porque la VRAM tiene límite.
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        # PASO 1: Borrar gradientes acumulados del batch anterior.
        # PyTorch ACUMULA gradientes por defecto, si no los borramos
        # se sumarían a los del batch anterior y los pesos se actualizarían
        # con información incorrecta.
        optimizer.zero_grad()

        # PASO 2: Forward pass — los datos fluyen a través de la red
        # y obtenemos las predicciones del batch actual.
        predicciones = model(X_batch)

        # PASO 3: Calcular el error (Loss) entre predicciones y valores reales.
        loss = criterion(predicciones, y_batch)

        # PASO 4: Backward pass — PyTorch calcula automáticamente el gradiente
        # de la pérdida respecto a CADA parámetro de la red (regla de la cadena).
        # Esto indica en qué dirección y cuánto hay que mover cada peso.
        loss.backward()

        # PASO 5: El optimizador actualiza los pesos usando los gradientes
        # calculados en el backward pass.
        optimizer.step()

        # Acumulamos la pérdida para calcular el promedio de la época.
        # .item() extrae el valor escalar del tensor (necesario para operar en CPU)
        train_loss_acumulado += loss.item()

        # MAE de entrenamiento: permite comparar visualmente con el MAE de validación
        # y detectar overfitting (si train MAE baja pero val MAE sube, hay sobreajuste)
        train_mae_acumulado += torch.mean(torch.abs(predicciones - y_batch)).item()

    # Pérdida media de todos los batches de entrenamiento en esta época
    train_loss_media = train_loss_acumulado / len(loader_train)
    train_mae_media  = train_mae_acumulado  / len(loader_train)


    # -------------------------------------------------------------------------
    # FASE 2: VALIDACIÓN (modo eval)
    # -------------------------------------------------------------------------
    # model.eval() activa el modo evaluación:
    # - Dropout está DESACTIVADO (todas las neuronas activas → predicciones estables)
    # - BatchNorm usa estadísticas fijas
    model.eval()
    val_loss_acumulado = 0.0
    val_mae_acumulado  = 0.0
    val_rmse_acumulado = 0.0

    # Para R² necesitamos acumular TODAS las predicciones y reales del epoch completo
    # No podemos calcularlo batch a batch porque R² necesita la media global
    todas_preds  = []
    todos_reales = []

    # torch.no_grad() desactiva el cálculo de gradientes durante la validación.
    # No necesitamos gradientes porque no vamos a actualizar pesos.
    # Ahorra memoria en GPU y acelera el cálculo ~2x.
    with torch.no_grad():
        for X_batch, y_batch in loader_val:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            predicciones = model(X_batch)

            # MSE en euros directos (también sirve como criterio de Early Stopping)
            val_loss_acumulado += criterion(predicciones, y_batch).item()

            # MAE, RMSE y R² directamente en euros — sin conversión
            val_mae_acumulado  += torch.mean(torch.abs(predicciones - y_batch)).item()
            val_rmse_acumulado += torch.sqrt(torch.mean((predicciones - y_batch) ** 2)).item()

            todas_preds.extend(predicciones.cpu().numpy().flatten())
            todos_reales.extend(y_batch.cpu().numpy().flatten())


    val_loss_media = val_loss_acumulado / len(loader_val)
    val_mae_media  = val_mae_acumulado  / len(loader_val)
    val_rmse_media = val_rmse_acumulado / len(loader_val)

        # R² (Coeficiente de Determinación):
        # Mide qué porcentaje de la variación del precio explica el modelo.
        # R² = 1 - (suma errores cuadráticos / varianza total del target)
        # R²=1.0 → predicción perfecta
        # R²=0.0 → el modelo no es mejor que predecir siempre la media
        # R²<0.0 → el modelo es peor que predecir siempre la media
    todas_preds  = np.array(todas_preds)
    todos_reales = np.array(todos_reales)
    ss_res = np.sum((todos_reales - todas_preds) ** 2)
    ss_tot = np.sum((todos_reales - np.mean(todos_reales)) ** 2)
    r2 = 1 - (ss_res / ss_tot)

        # Guardamos en el historial para las gráficas posteriores
    historial_train_loss.append(train_loss_media)
    historial_train_mae.append(train_mae_media)
    historial_val_loss.append(val_loss_media)
    historial_val_mae.append(val_mae_media)
    historial_val_rmse.append(val_rmse_media)
    historial_r2.append(r2)


    # -------------------------------------------------------------------------
    # LÓGICA DE EARLY STOPPING
    # -------------------------------------------------------------------------
    if val_loss_media < mejor_val_loss:
        # El modelo mejoró: actualizamos el mejor loss, reseteamos el contador
        # y guardamos una copia de los pesos actuales en memoria.
        # .state_dict() devuelve un diccionario con todos los pesos y biases.
        # copy.deepcopy garantiza que guardamos una copia independiente,
        # no una referencia que se sobreescribiría en la siguiente época.
        mejor_val_loss     = val_loss_media
        epocas_sin_mejora  = 0
        mejor_modelo_pesos = copy.deepcopy(model.state_dict())
        mejor_r2           = r2
        estado = "✅ Mejor"
    else:
        epocas_sin_mejora += 1
        estado = f"⏳ {epocas_sin_mejora}/{PATIENCE}"

    # Imprimir progreso cada 5 épocas (o en la primera)
    if epoca % 5 == 0 or epoca == 1:
        print(f"{epoca:<8} {train_loss_media:>10.2f} {train_mae_media:>10.2f} {val_loss_media:>10.2f} {val_mae_media:>10.2f} {val_rmse_media:>10.2f} {r2:>8.4f} {estado:>10}")

    # Condición de parada: si llevamos PATIENCE épocas sin mejorar, paramos
    if epocas_sin_mejora >= PATIENCE:
        print(f"\n⛔ Early Stopping activado en época {epoca}.")
        print(f"   El modelo no mejoró en {PATIENCE} épocas consecutivas.")
        break


# =============================================================================
# RESTAURAR LOS MEJORES PESOS Y GUARDAR EL MODELO
# =============================================================================
# Cargamos los pesos de la época donde el modelo tuvo menor val_loss.
# Sin esto, el modelo final tendría los pesos de la última época,
# que podrían estar sobreajustados.
model.load_state_dict(mejor_modelo_pesos)

# Guardamos el modelo entrenado a disco para usarlo en la API.
# state_dict() es preferible a guardar el modelo entero porque es
# más portable y no depende de la estructura de clases del proyecto.
import os

MEJOR_MODELO_PATH = '../data/airbnb_mlp_PriceCAP_NoLog.pt'

# Cargar el mejor R² histórico guardado en disco (si existe)
mejor_r2_historico = -float('inf')
if os.path.exists(MEJOR_MODELO_PATH.replace('.pt', '_meta_NoLog.txt')):
    with open(MEJOR_MODELO_PATH.replace('.pt', '_meta_NoLog.txt'), 'r') as f:
        mejor_r2_historico = float(f.read())

# Solo guardar si el nuevo modelo supera al histórico
if mejor_r2 > mejor_r2_historico:
    torch.save(mejor_modelo_pesos, MEJOR_MODELO_PATH)
    with open(MEJOR_MODELO_PATH.replace('.pt', '_meta.txt'), 'w') as f:
        f.write(str(mejor_r2))
    print(f"✅ Modelo guardado — nuevo R²: {mejor_r2:.4f} (anterior: {mejor_r2_historico:.4f})")
else:
    print(f"⏭️  Modelo NO guardado — R² actual {mejor_r2:.4f} no supera el histórico {mejor_r2_historico:.4f}")

print(f"\n{'='*60}")
#print(f"✅ Modelo guardado en '../data/airbnb_mlp_PriceCAP.pt'")
print(f"   Mejor Val MSE:  {mejor_val_loss:.2f}")
print(f"   Mejor Val MAE:  {min(historial_val_mae):.2f} €/noche")
print(f"   Mejor Val RMSE: {min(historial_val_rmse):.2f} €/noche")
print(f"   Mejor R²:       {mejor_r2:.4f}  →  El modelo explica el {mejor_r2*100:.1f}% de la variación de precios")

Iniciando entrenamiento en: cuda
Época     Train MSE  Train MAE    Val MSE    Val MAE   Val RMSE       R²     Estado
------------------------------------------------------------------------------------------
1          16673.81     112.56   15518.54     109.95     124.15  -3.3003    ✅ Mejor
5          15028.86     106.87   13925.03     103.99     117.60  -2.8600    ✅ Mejor
10          9959.19      87.28    9509.37      85.78      97.22  -1.6370    ✅ Mejor
15          5560.60      61.63    5216.17      57.97      71.93  -0.4464    ✅ Mejor
20          3125.22      41.84    2944.76      38.61      53.54   0.1744    ✅ Mejor
25          2049.38      32.10    2113.22      30.04      45.05   0.4071    ✅ Mejor
30          1778.74      29.94    2010.24      29.17      44.12   0.4405     ⏳ 1/30
35          1706.25      29.10    1775.02      27.31      41.39   0.5076    ✅ Mejor
40          1571.49      27.13    1673.84      25.57      40.18   0.5322     ⏳ 1/30
45          1458.07      26.46    16

Tras múltiples ensayos y pruebas la RRNN sin transformación logarítmica es la que arroja mejores resultados ya que ofrece sólidos resultados entre 55-58% y la RRNN con trnasformación logarítmica entre 50-55%. Presenta actualmente un error de 24.84€/noche que esperamos ver reducido a la vez que aumenta el accuracy cuando unamos esta RRNN con la RRNN convolucional de las imágenes.